# TSG-NIDS v5: final experiments for the IEEE Access paper

**Method (frozen).** Stage 1 is a regularised LightGBM gate. Stage 2 is a stacked ensemble (random forest, extra
trees, LightGBM; XGBoost meta-learner) whose classes are the attack types **and benign**; it is trained on attack
flows, synthetic flows of rare classes from a **per-class CTGAN** (class-wise rounding and clipping), and benign
hard negatives (benign training flows with an out-of-fold gate score of at least 0.10). The gate threshold τ is
chosen on validation. SMOTE is reported as the cheaper equivalent generator (R-SMOTE).

**What v5 adds to v4 (the pre-writing checklist)**

| # | Item | Where |
|---|---|---|
| 1 | Fair single-stage comparison: S1-STACK and S1-LGBM thresholded on their attack score 1 − P(benign), every system scored at 0.5 / 1 / 2 % false alarms, S1-STACK with five seeds | Sections 11, 14 |
| 2 | Threshold rule with a safety margin: τ keeps the validation false-alarm rate at or below `FPR_MARGIN` × budget | Sections 10, 14 |
| 3 | Hard-negative sensitivity: score threshold 0.05 / 0.10 / 0.25 and cap 100k / 200k / all, one at a time | Section 17 |
| 4 | Configuration frozen: per-class CTGAN is the proposed generator (`PROPOSED_GEN`), nothing is selected on test | Sections 0, 10 |
| 5 | One run produces every number; training and inference times measured on this machine, Stage 2 timed on gated flows only | Section 20 |
| 6 | Published methods re-run under this protocol: SMOTE + RF / DT (Karatas et al., 2020), DSSTE + RF / XGBoost (Liu et al., 2021) | Section 12 |
| 7 | Where the false alarms go, attack-only vs benign-class Stage 2, at the same budget | Section 16 |

**Reuse.** Keep `OUT_DIR` pointing at the v3/v4 folder: the split, features, gate, Stage-2 models, synthetic pools and
seed models are loaded from `checkpoints`, so only the new parts are trained. Model outputs do not depend on the
machine, but times do: Section 20 re-times every system on the current runtime. For a fully fresh run on one
machine set `FORCE_RETRAIN = True` (about three times longer). Results go to `results_v5`.

**Machine.** Run both datasets on the same runtime type (same CPU count, GPU or no GPU), so the timing tables of the
two datasets can be compared. The machine is written to `environment.json`.

**Run time** (estimate) with v4 checkpoints on a 44-vCPU high-memory runtime: about 2–3 h for CIC-IDS2017 and 4–6 h
for CSE-CIC-IDS2018. The slowest parts are the S1-STACK seed models, the hard-negative grid, SMOTE + RF on the full
training split (about 7.5 million rows on CSE-CIC-IDS2018, 10–15 GB of memory) and the timing section; each has a
switch in Section 0 (`S1_SEED_MODELS`, `RUN_HN_SENS`, `RUN_PUBLISHED`, `RUN_TIMING`).

In [ ]:
%pip install -q numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 scipy==1.16.3 lightgbm==4.7.0 xgboost==3.4.1 imbalanced-learn==0.14.2 matplotlib==3.10.0 ctgan==0.12.1 sdv==1.38.5 kagglehub joblib pyarrow openpyxl psutil

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab: set NIDS_DATASET, NIDS_DATA_PATH / CICIDS2017_DIR and NIDS_OUT_DIR before the next cell.")

## 0. Configuration

In [ ]:
import os, sys, re, glob, json, time, platform, shutil, hashlib, warnings
os.environ.setdefault("TQDM_DISABLE", "1")     # silence generator progress bars

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder, label_binarize, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, matthews_corrcoef,
                             roc_auc_score, average_precision_score, confusion_matrix)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from scipy.special import expit
from scipy.stats import binomtest, chi2, ks_2samp

import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- dataset ----------------
DATASET        = os.environ.get("NIDS_DATASET", "cic-ids2017")        # "cic-ids2017" or "cse-cic-ids2018"
DATA_2018_CSV  = os.environ.get("NIDS_DATA_PATH", "/content/drive/MyDrive/Datasets/combined_dataset.csv")
KAGGLE_2017    = "chethuhn/network-intrusion-dataset"
CICIDS2017_DIR = os.environ.get("CICIDS2017_DIR", "")   # folder holding the 8 CSVs; empty = download with kagglehub
OUT_DIR        = os.environ.get("NIDS_OUT_DIR", f"/content/drive/MyDrive/Datasets/NIDS_v3_{DATASET}")  # v3 folder: reuse
CKPT_DIR, RES_DIR = os.path.join(OUT_DIR, "checkpoints"), os.path.join(OUT_DIR, "results_v5")
FIG_DIR = os.path.join(RES_DIR, "figures")

BENIGN_LABEL = {"cic-ids2017": "BENIGN", "cse-cic-ids2018": "Benign"}[DATASET]
LABEL_COL = "Label"
ID_COLS = ["Flow ID", "Src IP", "Source IP", "Dst IP", "Destination IP", "Src Port", "Source Port", "Timestamp"]

# ---------------- split ----------------
SEED, TEST_SIZE, VAL_SIZE = 42, 0.20, 0.15

# ---------------- feature selection ----------------
IG_SAMPLE, IG_TOP_K, CORR_THRESHOLD = 300_000, 45, 0.90   # published method (gate)
MI_PER_CLASS, PS_TOP_K, PS_CORR = 20_000, 45, 0.95        # Stage-2 features

# ---------------- augmentation ----------------
MINORITY_CUTOFF = 10_000
AUG_TARGET      = 25_000
AUG_MIN_REAL    = 50
SMOTE_K         = 5
DISCRETE_MAX_UNIQUE = 10     # integer feature columns with at most this many values are modelled as categorical
TVAE_BALANCE_CAP    = 5_000  # TVAE has no conditional sampling: it is trained on a class-balanced table
CTGAN_PC_MIN_STEPS  = 3_000  # per-class CTGAN: at least this many generator updates per class
QUALITY_SYN_ROWS    = 2_000

# ---------------- Stage 2 with a benign class ----------------
R_METHODS           = ["none", "smote", "ctgan", "ctgan_pc", "tvae"]   # augmentation variants of the benign-class Stage 2
PROPOSED_GEN        = "ctgan_pc"      # frozen (item 4): validation picked per-class CTGAN on both datasets in v4
PROPOSED_CANDIDATES = ["ctgan", "ctgan_pc"]   # only used for the tau = 0.45 ablation rows
HN_TAU, HN_CAP      = 0.10, 200_000   # benign hard negatives: out-of-fold gate score >= HN_TAU, at most HN_CAP flows
GATE_HP_SET         = "stable"        # "stable" = v4 regularised gate; "published" = v3 gate (reuses v3 gate checkpoints)
VAL_GATE_SOURCE     = "auto"          # validation gate scores: "auto", "fit_gate" or "oof"
GATE_CHECK_TAUS     = [0.45, 0.65, 0.85]
GATE_CHECK_RATIO, GATE_CHECK_ABS = 2.0, 0.002

# ---------------- v3 systems kept for reference ----------------
RUN_V3_SYSTEMS = True
AUG_METHODS_V3 = ["none", "smote", "ctgan", "tvae", "copulagan"]
VAL_METHODS_V3 = ["none", "smote", "ctgan"]
RUN_CHAIN      = True        # T-CTGAN-PS-CC, the step between v3 and v4 in the ablation chain

# ---------------- thresholds ----------------
TAU_MAIN       = 0.45
TAU_GRID       = np.round(np.arange(0.05, 0.951, 0.01), 2)            # published rule and test sweeps (as v4)
TAU_GRID_VAL   = np.unique(np.r_[np.round(np.arange(0.01, 0.991, 0.01), 2), 0.995, 0.999])  # every rule chosen on validation
MAX_BENIGN_FPR = 0.02        # budget of the main rule: final benign false alarms
FPR_CAPS       = [0.005, 0.01, 0.02]   # false-alarm budgets of the budget tables (item 1)
FPR_MARGIN     = 0.8         # item 2: tau keeps validation false alarms at or below FPR_MARGIN x budget (1.0 = v4 rule)
FA_BUDGET      = 0.01        # budget used for the extra rows of the results table and for item 7

# ---------------- experiments ----------------
RUN_FLAT       = ["rf", "xgb", "lgbm", "stack"]
SEEDS_REPEAT   = [42, 7, 21, 63, 99]
SEED_SYSTEMS   = ["T-SMOTE", "R-NONE", "R-SMOTE", "PROPOSED"]
N_BOOT         = 1000
FORCE_RETRAIN  = False
SAVE_MODELS    = False

# ---------------- item 1: thresholded single-stage baselines ----------------
S1_THRESHOLDED = ["stack", "lgbm"]     # fitted on the fit part as well, so tau can be chosen on validation
S1_SEED_MODELS = ["stack"]             # retrained with every seed in SEEDS_REPEAT
BUDGET_SYSTEMS = ["S1-STACK", "S1-LGBM", "T-NONE", "T-SMOTE", "T-CTGAN", "R-NONE", "R-SMOTE", "R-CTGAN", "R-CTGAN-PC", "R-TVAE"]
BUDGET_ROWS    = ["S1-STACK", "S1-LGBM", "T-SMOTE", "R-NONE", "R-SMOTE", "PROPOSED"]   # also added to the results table at FA_BUDGET
FA_PAIRS       = [("T-SMOTE", "PROPOSED"), ("T-NONE", "R-NONE")]   # item 7: attack-only vs benign-class Stage 2

# ---------------- item 3: hard-negative sensitivity (one setting changed at a time) ----------------
RUN_HN_SENS    = True
HN_SENS_TAUS   = [0.05, 0.10, 0.25]    # with HN_CAP fixed
HN_SENS_CAPS   = [100_000, 200_000, None]   # with HN_TAU fixed; None = every eligible flow

# ---------------- item 5: timing on this machine ----------------
RUN_TIMING     = True
TIMING_SYSTEMS = ["S1-RF", "S1-XGB", "S1-LGBM", "S1-STACK", "T-SMOTE", "R-SMOTE", "R-CTGAN-PC"]
TIME_GENERATOR = True                  # also re-time the per-class CTGAN fit

# ---------------- item 6: published methods under this protocol ----------------
RUN_PUBLISHED  = True
PUBLISHED      = ["KARATAS-RF", "KARATAS-DT", "DSSTE-RF", "DSSTE-XGB"]
KARATAS_RATIO  = 10       # SMOTE raises every attack class to (largest class) / KARATAS_RATIO; the paper reports an
                          # imbalance ratio of 9.98 after SMOTE
DSSTE_PARAMS   = dict(train_cap_benign=100_000, train_cap_attack=20_000,   # DSSTE was run on a 100,000-flow subset
                      enn_k=3, enn_kind="all",      # imbalanced-learn ENN defaults
                      compress=0.1,                 # majority flows in the difficult set -> 10 % as k-means centres
                      zoom_k=5)                     # minority difficult flows: zoom_k copies, continuous columns x U(1-1/k, 1+1/k)
                      # check enn_k, compress and zoom_k against Liu et al. (2021), Algorithm 1, before reporting

HP = {
    "gate_lgbm":   dict(n_estimators=400, num_leaves=64, learning_rate=0.1, colsample_bytree=0.8),
    "gate_stable": dict(min_child_weight=1.0, reg_lambda=1.0),          # added when GATE_HP_SET = "stable"
    "s2_rf":       dict(n_estimators=200),
    "s2_et":       dict(n_estimators=200),
    "s2_lgbm":     dict(n_estimators=300, num_leaves=64),
    "s2_meta_xgb": dict(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist"),
    "stack_cv_folds": 3, "gate_oof_folds": 3,
    "flat_rf":     dict(n_estimators=200, class_weight="balanced_subsample"),
    "flat_xgb":    dict(n_estimators=200, max_depth=8, learning_rate=0.1, tree_method="hist"),
    "flat_lgbm":   dict(n_estimators=300, num_leaves=64, learning_rate=0.1, min_child_weight=1.0, reg_lambda=1.0),
    "ctgan":       dict(epochs=300, batch_size=500, generator_dim=(256, 256), discriminator_dim=(256, 256), pac=10),
    "ctgan_pc":    dict(epochs=300, batch_size=500, generator_dim=(256, 256), discriminator_dim=(256, 256), pac=10),
    "tvae":        dict(epochs=300, batch_size=500),
    "copulagan":   dict(epochs=300, batch_size=500),
}

for d in (CKPT_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

## 1. Helpers

In [ ]:
def ckpt(n): return os.path.join(CKPT_DIR, n)
def res(n):  return os.path.join(RES_DIR, n)
def fig(n):  return os.path.join(FIG_DIR, n)
def have(*names): return (not FORCE_RETRAIN) and all(os.path.exists(ckpt(n)) for n in names)
def mcode(m): return m.upper().replace("_", "-")

def save_json(obj, path):
    def conv(o):
        if isinstance(o, np.bool_): return bool(o)
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=conv)

def load_json(path):
    with open(path) as f:
        return json.load(f)

class Timer:
    def __enter__(self): self.t0 = time.perf_counter(); return self
    def __exit__(self, *a): self.s = time.perf_counter() - self.t0

GAN_METHODS = {"ctgan", "tvae", "copulagan", "ctgan_pc"}
if "copulagan" in AUG_METHODS_V3:
    try:
        import sdv  # noqa: F401
    except ImportError:
        print("sdv is not installed: CopulaGAN is skipped (pip install sdv to include it).")
        AUG_METHODS_V3.remove("copulagan")
VAL_METHODS_V3 = [m for m in VAL_METHODS_V3 if m in AUG_METHODS_V3]
for m in PROPOSED_CANDIDATES:
    if m not in R_METHODS:
        R_METHODS.append(m)
assert PROPOSED_GEN in R_METHODS and PROPOSED_GEN != "auto", "v5 uses a frozen generator"

def cpu_model():
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"

def gpu_name():
    try:
        import torch
        return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    except Exception:
        return "none"

MACHINE = {"cpu_model": cpu_model(), "cpu_count": os.cpu_count(), "gpu": gpu_name()}
def same_machine(info): return all(info.get(k) == v for k, v in MACHINE.items())

def log_environment():
    import sklearn, lightgbm, xgboost, imblearn, scipy
    env = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_count": os.cpu_count(),
           "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
           "lightgbm": lightgbm.__version__, "xgboost": xgboost.__version__, "imbalanced-learn": imblearn.__version__,
           "scipy": scipy.__version__, "matplotlib": matplotlib.__version__}
    for mod in ("torch", "ctgan", "sdv", "kagglehub"):
        try:
            env[mod] = __import__(mod).__version__
        except Exception:
            pass
    try:
        import torch; env["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    except Exception:
        pass
    try:
        import psutil; env["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        pass
    env["config"] = {k: globals()[k] for k in [
        "DATASET", "SEED", "TEST_SIZE", "VAL_SIZE", "IG_SAMPLE", "IG_TOP_K", "CORR_THRESHOLD", "MI_PER_CLASS",
        "PS_TOP_K", "PS_CORR", "MINORITY_CUTOFF", "AUG_TARGET", "AUG_MIN_REAL", "SMOTE_K", "CTGAN_PC_MIN_STEPS",
        "R_METHODS", "PROPOSED_GEN", "PROPOSED_CANDIDATES", "HN_TAU", "HN_CAP", "GATE_HP_SET", "VAL_GATE_SOURCE",
        "GATE_CHECK_TAUS", "GATE_CHECK_RATIO", "GATE_CHECK_ABS", "RUN_V3_SYSTEMS", "AUG_METHODS_V3", "VAL_METHODS_V3",
        "RUN_CHAIN", "TAU_MAIN", "MAX_BENIGN_FPR", "FPR_CAPS", "FPR_MARGIN", "FA_BUDGET", "RUN_FLAT", "SEEDS_REPEAT",
        "SEED_SYSTEMS", "N_BOOT", "S1_THRESHOLDED", "S1_SEED_MODELS", "BUDGET_SYSTEMS", "RUN_HN_SENS", "HN_SENS_TAUS",
        "HN_SENS_CAPS", "RUN_TIMING", "TIMING_SYSTEMS", "TIME_GENERATOR", "RUN_PUBLISHED", "PUBLISHED", "KARATAS_RATIO",
        "DSSTE_PARAMS", "BUDGET_ROWS", "FA_PAIRS"]}
    env["config"]["TAU_GRID_VAL"] = [float(TAU_GRID_VAL.min()), float(TAU_GRID_VAL.max()), int(len(TAU_GRID_VAL))]
    env["machine"] = MACHINE
    env["hyperparameters"] = HP
    save_json(env, res("environment.json"))
    print({k: v for k, v in env.items() if k not in ("config", "hyperparameters")})

log_environment()

## 2. Data
Unchanged from v3 (the split is loaded from the v3 checkpoints when present). CIC-IDS2017 is downloaded with
`kagglehub.dataset_download`; if Kaggle asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab
secrets, or download the files yourself and set `CICIDS2017_DIR`.

Cleaning (both datasets): strip column names, drop repeated header rows, turn ±inf into missing, drop rows with
missing values, drop exact duplicate rows, drop identifier columns. CIC-IDS2017 web-attack labels contain a broken
character ("Web Attack � Brute Force"); they are renamed "Web Attack - Brute Force" etc.

In [ ]:
def read_raw():
    if DATASET == "cse-cic-ids2018":
        return pd.read_csv(DATA_2018_CSV, low_memory=False), [DATA_2018_CSV]
    folder = CICIDS2017_DIR
    if not folder:
        import kagglehub
        folder = kagglehub.dataset_download(KAGGLE_2017)
    files = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
    assert files, f"no CSV files found under {folder}"
    frames = []
    for f in files:
        d = pd.read_csv(f, low_memory=False, encoding="utf-8", encoding_errors="replace")
        d.columns = d.columns.str.strip()
        frames.append(d)
        print(f"  {os.path.basename(f)}: {d.shape}")
    return pd.concat(frames, ignore_index=True), files

def load_split():
    files = ["X_train.parquet", "X_test.parquet", "y_train.npy", "y_test.npy", "label_encoder.joblib", "prep_log.json"]
    if have(*files):
        print("[data] checkpoints found — loading")
        return (pd.read_parquet(ckpt(files[0])), pd.read_parquet(ckpt(files[1])), np.load(ckpt(files[2])),
                np.load(ckpt(files[3])), joblib.load(ckpt(files[4])), load_json(ckpt(files[5])))
    log = {"dataset": DATASET}
    df, src = read_raw()
    df.columns = df.columns.str.strip()
    log["source_files"] = [os.path.basename(s) for s in src]
    log["rows_raw"], log["cols_raw"] = int(len(df)), int(df.shape[1])
    log["input_fingerprint"] = hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()
    df = df[df[LABEL_COL].astype(str).str.strip() != LABEL_COL]
    lab = df[LABEL_COL].astype(str).str.strip()
    df[LABEL_COL] = lab.str.replace(r"^Web Attack[^A-Za-z]+", "Web Attack - ", regex=True)
    for c in [c for c in df.columns if c != LABEL_COL and c not in ID_COLS and df[c].dtype == object]:
        df[c] = pd.to_numeric(df[c], errors="coerce")          # stray text in numeric columns
    df = df.replace([np.inf, -np.inf], np.nan)
    n0 = len(df); df = df.dropna();          log["rows_removed_nan_inf"] = int(n0 - len(df))
    n0 = len(df); df = df.drop_duplicates(); log["rows_removed_duplicates"] = int(n0 - len(df))
    df = df.drop(columns=[c for c in ID_COLS if c in df.columns])
    log["rows_clean"] = int(len(df))
    le = LabelEncoder()
    y = le.fit_transform(df[LABEL_COL])
    X = df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number])
    del df
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
    X_train, X_test = X_train.reset_index(drop=True), X_test.reset_index(drop=True)
    log["constant_columns_in_train"] = [c for c in X_train.columns if X_train[c].nunique() <= 1]
    log["n_features"] = int(X_train.shape[1])
    log["rows_train"], log["rows_test"] = int(len(X_train)), int(len(X_test))
    X_train.to_parquet(ckpt(files[0])); X_test.to_parquet(ckpt(files[1]))
    np.save(ckpt(files[2]), y_train); np.save(ckpt(files[3]), y_test)
    joblib.dump(le, ckpt(files[4])); save_json(log, ckpt(files[5]))
    return X_train, X_test, y_train, y_test, le, log


X_train, X_test, y_train, y_test, le, PREP = load_split()
save_json(PREP, res("preprocessing_log.json"))
CLASS_NAMES  = list(le.classes_)
K            = len(CLASS_NAMES)
BENIGN       = CLASS_NAMES.index(BENIGN_LABEL)
ATTACK_CODES = np.array([c for c in range(K) if c != BENIGN])
ATTACK_NAMES = [CLASS_NAMES[c] for c in ATTACK_CODES]
A            = len(ATTACK_CODES)
R_CODES      = np.r_[ATTACK_CODES, BENIGN]          # Stage 2 with a benign class: compact index A = benign
TO_COMPACT   = np.full(K, -1); TO_COMPACT[ATTACK_CODES] = np.arange(A)
INFIL        = next((c for c, n in enumerate(CLASS_NAMES) if "infil" in n.lower()), None)
train_counts = np.bincount(y_train, minlength=K)
MINORITY_FULL = np.array([c for c in range(K) if c != BENIGN and train_counts[c] < MINORITY_CUTOFF])
AUG_CODES    = np.array([c for c in MINORITY_FULL if train_counts[c] >= AUG_MIN_REAL])
AUG_COMPACT  = TO_COMPACT[AUG_CODES]
print({k: v for k, v in PREP.items() if k != "source_files"})
print("Minority classes (<%d training flows):" % MINORITY_CUTOFF, [CLASS_NAMES[c] for c in MINORITY_FULL])
print("Augmented classes:", [CLASS_NAMES[c] for c in AUG_CODES])

if have("fit_idx.npy", "val_idx.npy"):
    fit_idx, val_idx = np.load(ckpt("fit_idx.npy")), np.load(ckpt("val_idx.npy"))
else:
    fit_idx, val_idx = train_test_split(np.arange(len(y_train)), test_size=VAL_SIZE, stratify=y_train, random_state=SEED)
    fit_idx, val_idx = np.sort(fit_idx), np.sort(val_idx)
    np.save(ckpt("fit_idx.npy"), fit_idx); np.save(ckpt("val_idx.npy"), val_idx)
all_idx = np.arange(len(y_train))
y_val = y_train[val_idx]
DIST = pd.DataFrame({"train": train_counts, "fit": np.bincount(y_train[fit_idx], minlength=K),
                     "validation": np.bincount(y_val, minlength=K), "test": np.bincount(y_test, minlength=K)},
                    index=CLASS_NAMES)
DIST["augmented"] = DIST.index.isin([CLASS_NAMES[c] for c in AUG_CODES])
DIST.sort_values("train", ascending=False).to_csv(res("table_class_distribution.csv"), index_label="class")
print(DIST.sort_values("train", ascending=False).to_string())

## 3. Feature selection (gate and v3 Stage 2)
Published method on the training split: mutual information on a random 300,000-flow sample, top 45, then a feature
is dropped if |Pearson r| > 0.90 with a higher-ranked one. The gate uses this set. `per_stage` (v3 feature ablation):
mutual information on class-balanced attack flows from the fit part, top 45, greedy correlation pruning at 0.95.
The v4 Stage-2 set (`per_stage_r`) is chosen in Section 6, after the gate, because it needs the benign hard negatives.

In [ ]:
def mi_scores(X, y):
    return pd.Series(mutual_info_classif(X, y, random_state=SEED, discrete_features=False, n_jobs=-1),
                     index=X.columns).sort_values(ascending=False)

def greedy_prune(Xs, ranked, thr):
    corr = Xs[ranked].corr().abs().fillna(0.0)
    kept = []
    for f in ranked:
        if all(corr.loc[f, k] <= thr for k in kept):
            kept.append(f)
    return kept

def balanced_rows(groups, rng):
    return np.sort(np.concatenate([g if len(g) <= MI_PER_CLASS else rng.choice(g, MI_PER_CLASS, replace=False)
                                   for g in groups if len(g)]))

def select_features():
    if have("features.json"):
        return load_json(ckpt("features.json"))
    idx = np.random.RandomState(SEED).choice(len(X_train), min(len(X_train), IG_SAMPLE), replace=False)
    ig = mi_scores(X_train.iloc[idx], y_train[idx])
    top = ig.head(IG_TOP_K).index.tolist()
    upper = X_train[top].corr().abs().where(np.triu(np.ones((len(top), len(top))), k=1).astype(bool))
    dropped = {c: upper[c][upper[c] > CORR_THRESHOLD].index[0] for c in top if (upper[c] > CORR_THRESHOLD).any()}
    feats = {"published": [f for f in top if f not in dropped]}
    att = fit_idx[y_train[fit_idx] != BENIGN]
    bi = balanced_rows([att[y_train[att] == c] for c in ATTACK_CODES], np.random.RandomState(SEED))
    ig2 = mi_scores(X_train.iloc[bi], y_train[bi])
    feats["per_stage"] = greedy_prune(X_train.iloc[bi], ig2.head(PS_TOP_K).index.tolist(), PS_CORR)
    tab = pd.DataFrame({"feature": X_train.columns, "mi_published": ig.reindex(X_train.columns).values,
                        "mi_stage2_balanced": ig2.reindex(X_train.columns).values})
    tab["rank_published"] = tab.mi_published.rank(ascending=False, method="min").astype(int)
    tab["selected_published"] = tab.feature.isin(feats["published"])
    tab["dropped_correlated_with"] = tab.feature.map(dropped).fillna("")
    tab["selected_per_stage"] = tab.feature.isin(feats["per_stage"])
    tab.sort_values("rank_published").to_csv(ckpt("table_feature_selection.csv"), index=False)
    save_json(feats, ckpt("features.json"))
    return feats


FEATS = select_features()
F_PUB = FEATS["published"]
GEN_COLS = list(dict.fromkeys(F_PUB + FEATS["per_stage"]))      # v3 generator columns
for src in (ckpt("table_feature_selection.csv"), os.path.join(OUT_DIR, "results", "table_feature_selection.csv")):
    if os.path.exists(src):
        shutil.copy(src, res("table_feature_selection.csv")); break
print(f"published: {len(F_PUB)} features; per-stage (attack flows): {len(FEATS['per_stage'])}")

## 4. Model builders

In [ ]:
def gate_params():
    hp = dict(HP["gate_lgbm"])
    if GATE_HP_SET == "stable":
        hp.update(HP["gate_stable"])
    return hp

def spw(yb): return float((yb == 0).sum() / max((yb == 1).sum(), 1))

def build_gate(seed, w):
    return LGBMClassifier(**gate_params(), scale_pos_weight=w, random_state=seed, n_jobs=-1, verbose=-1)

def build_stack(n_classes, seed):
    level0 = [("rf",   RandomForestClassifier(**HP["s2_rf"], n_jobs=-1, random_state=seed)),
              ("et",   ExtraTreesClassifier(**HP["s2_et"], n_jobs=-1, random_state=seed)),
              ("lgbm", LGBMClassifier(**HP["s2_lgbm"], random_state=seed, n_jobs=-1, verbose=-1))]
    meta = XGBClassifier(**HP["s2_meta_xgb"], objective="multi:softprob", num_class=n_classes,
                         random_state=seed, n_jobs=-1, eval_metric="mlogloss")
    cv = StratifiedKFold(HP["stack_cv_folds"], shuffle=True, random_state=seed)
    return StackingClassifier(level0, meta, stack_method="predict_proba", cv=cv, n_jobs=1)

FLAT_BUILDERS = {
    "rf":    lambda s=SEED: RandomForestClassifier(**HP["flat_rf"], n_jobs=-1, random_state=s),
    "xgb":   lambda s=SEED: XGBClassifier(**HP["flat_xgb"], objective="multi:softprob", num_class=K, random_state=s,
                                          n_jobs=-1, eval_metric="mlogloss"),
    "lgbm":  lambda s=SEED: LGBMClassifier(**HP["flat_lgbm"], random_state=s, n_jobs=-1, verbose=-1),
    "stack": lambda s=SEED: build_stack(K, s),
}
FLAT_CKPT = {"rf": "flat_rf", "xgb": "flat_xgb", "lgbm": "flat_lgbm4", "stack": "flat_stack"}   # lgbm refitted in v4
FLAT_LABELS = {"rf": "Random forest (single-stage)", "xgb": "XGBoost (single-stage)",
               "lgbm": "LightGBM (single-stage, regularised)", "stack": "Stacked ensemble (single-stage)"}

## 5. Stage 1: gate and gate check
The gate is trained on original flows only. Four sets of scores:
* **fit-part gate** (trained on the fit part) scores the validation split;
* **full gate** (trained on the whole training split) scores the test split;
* **out-of-fold, training split** (3 folds): published threshold rule and benign hard negatives for the full Stage 2;
* **out-of-fold, fit part** (3 folds): benign hard negatives for the fit-part Stage 2 (validation).

`GATE_HP_SET = "stable"` adds `min_child_weight = 1` and `reg_lambda = 1` to the published gate settings. The
largest absolute logit of every fit is logged: values in the thousands mean the boosting diverged.

**Gate check.** On the validation flows, the false-alarm rate of the fit-part gate is compared with that of the
out-of-fold scores (and with the test rate). If the fit-part gate flags more than `GATE_CHECK_RATIO` times as many
benign flows (plus `GATE_CHECK_ABS`) at any τ in `GATE_CHECK_TAUS`, validation uses the out-of-fold scores.

In [ ]:
GATE_TAG = "gate4" if GATE_HP_SET == "stable" else "gate"

def gate_fit_score(rows, X_eval):
    yb = (y_train[rows] != BENIGN).astype(int)
    g = build_gate(SEED, spw(yb))
    with Timer() as t_fit:
        g.fit(X_train.iloc[rows][F_PUB], yb)
    with Timer() as t_pred:
        raw = g.predict(X_eval[F_PUB], raw_score=True)
    return expit(raw), {"train_seconds": t_fit.s, "predict_seconds": t_pred.s, "rows": int(len(rows)),
                        "max_abs_logit": float(np.abs(raw).max())}, g

def gate_oof(rows):
    yb = (y_train[rows] != BENIGN).astype(int)
    out, mx = np.empty(len(rows)), 0.0
    for tr, te in StratifiedKFold(HP["gate_oof_folds"], shuffle=True, random_state=SEED).split(np.zeros(len(rows)), yb):
        g = build_gate(SEED, spw(yb))
        g.fit(X_train.iloc[rows[tr]][F_PUB], yb[tr])
        raw = g.predict(X_train.iloc[rows[te]][F_PUB], raw_score=True)
        out[te] = expit(raw); mx = max(mx, float(np.abs(raw).max()))
    return out, {"max_abs_logit": mx, "rows": int(len(rows))}

GN = {k: f"{GATE_TAG}_{k}.npy" for k in ("val", "test", "oof", "oof_fit")}
if have(GN["val"], GN["test"], GN["oof"], f"{GATE_TAG}_timing.json"):
    p_val, p_test, p_oof = (np.load(ckpt(GN[k])) for k in ("val", "test", "oof"))
    GATE_TIMING = load_json(ckpt(f"{GATE_TAG}_timing.json"))
else:
    p_val, t_v, _ = gate_fit_score(fit_idx, X_train.iloc[val_idx])
    p_test, t_t, gate = gate_fit_score(all_idx, X_test)
    p_oof, t_o = gate_oof(all_idx)
    GATE_TIMING = {"fit_part": t_v, "full": t_t, "oof": t_o}
    for k, a in (("val", p_val), ("test", p_test), ("oof", p_oof)):
        np.save(ckpt(GN[k]), a)
    save_json(GATE_TIMING, ckpt(f"{GATE_TAG}_timing.json"))
    if SAVE_MODELS: joblib.dump(gate, ckpt(f"{GATE_TAG}_full.joblib"))
if have(GN["oof_fit"]):
    p_oof_fit = np.load(ckpt(GN["oof_fit"]))
else:
    p_oof_fit, t_of = gate_oof(fit_idx)
    np.save(ckpt(GN["oof_fit"]), p_oof_fit); save_json(t_of, ckpt(f"{GATE_TAG}_oof_fit.json"))
P_OOF_FIT = np.full(len(y_train), np.nan); P_OOF_FIT[fit_idx] = p_oof_fit
print("gate:", GATE_HP_SET, gate_params())
print("largest |logit|:", {k: round(v.get("max_abs_logit", float("nan")), 1) for k, v in GATE_TIMING.items()})

# published threshold rule: maximise benign-vs-attack F1 on out-of-fold scores
yb_tr = (y_train != BENIGN).astype(int)
f1s = [2 * np.sum((p_oof >= t) & (yb_tr == 1)) / max(np.sum(p_oof >= t) + yb_tr.sum(), 1) for t in TAU_GRID]
TAU_PUB = float(TAU_GRID[int(np.argmax(f1s))])

# gate check
def rate(p, mask, t): return float((p[mask] >= t).mean()) if mask.any() else float("nan")
vb, vatt, tb, tatt = y_val == BENIGN, y_val != BENIGN, y_test == BENIGN, y_test != BENIGN
p_oof_val = p_oof[val_idx]
GCHK = pd.DataFrame([{"tau": t,
                      "fpr_validation_fit_gate": rate(p_val, vb, t), "fpr_validation_oof": rate(p_oof_val, vb, t),
                      "fpr_test_full_gate": rate(p_test, tb, t),
                      "recall_validation_fit_gate": rate(p_val, vatt, t), "recall_validation_oof": rate(p_oof_val, vatt, t),
                      "recall_test_full_gate": rate(p_test, tatt, t)} for t in np.round(np.arange(0.05, 0.951, 0.05), 2)])
GCHK.to_csv(res("table_gate_check.csv"), index=False)
failed = [t for t in GATE_CHECK_TAUS
          if rate(p_val, vb, t) > GATE_CHECK_RATIO * max(rate(p_oof_val, vb, t), rate(p_test, tb, t)) + GATE_CHECK_ABS]
VAL_SRC = VAL_GATE_SOURCE if VAL_GATE_SOURCE != "auto" else ("oof" if failed else "fit_gate")
p_val_used = p_val if VAL_SRC == "fit_gate" else p_oof_val
GATE_CHECK = {"gate_hp_set": GATE_HP_SET, "failed_at_tau": failed, "validation_scores": VAL_SRC,
              "auc_validation_fit_gate": roc_auc_score(vatt, p_val), "auc_validation_oof": roc_auc_score(vatt, p_oof_val),
              "auc_test_full_gate": roc_auc_score(tatt, p_test),
              "share_validation_benign_p_ge_0.99_fit_gate": rate(p_val, vb, 0.99),
              "share_validation_benign_p_ge_0.99_oof": rate(p_oof_val, vb, 0.99),
              "share_test_benign_p_ge_0.99": rate(p_test, tb, 0.99),
              "max_abs_logit": {k: v.get("max_abs_logit") for k, v in GATE_TIMING.items()}}
save_json(GATE_CHECK, res("gate_check.json"))
GATE_TABLE = pd.DataFrame([{"split": "test", "tau": t, "roc_auc": GATE_CHECK["auc_test_full_gate"],
                            "attack_recall": rate(p_test, tatt, t), "benign_fpr": rate(p_test, tb, t)}
                           for t in (TAU_PUB, TAU_MAIN, HN_TAU)])
GATE_TABLE.to_csv(res("table_gate.csv"), index=False)
print(GCHK[GCHK.tau.isin([0.1, 0.25, 0.45, 0.65, 0.85, 0.95])].round(4).to_string(index=False))
print({k: (round(v, 5) if isinstance(v, float) else v) for k, v in GATE_CHECK.items()})
print(f"validation uses {VAL_SRC} scores; tau by published rule: {TAU_PUB:.2f}")

## 6. Benign hard negatives and Stage-2 features
Hard negatives are benign training flows with an out-of-fold gate score of at least `HN_TAU` (0.10, the lowest τ
the v4 systems may use). They are the flows the gate would pass to Stage 2. If there are more than `HN_CAP`, a
random subset is kept. The full Stage 2 takes them from the training split, the fit-part Stage 2 from the fit part.

The v4 Stage-2 features (`per_stage_r`) are chosen on the fit part: mutual information on a class-balanced sample
of attack flows (≤ 20,000 per class) plus hard negatives (≤ 20,000), top 45, greedy correlation pruning at 0.95.

In [ ]:
HN_MAIN = (HN_TAU, HN_CAP)

def hn_tag(hn):
    t, cap = hn
    return f"{GATE_TAG}_hn{int(round(t * 100))}c" + ("all" if cap is None else f"{cap // 1000}k")

HN_TAG = hn_tag(HN_MAIN)          # same checkpoint names as v4 for the main setting

def hard_negatives(rows, part, seed, hn=None):
    t, cap = hn or HN_MAIN
    score = p_oof if part == "full" else P_OOF_FIT
    b = rows[(y_train[rows] == BENIGN) & (score[rows] >= t)]
    if cap is not None and len(b) > cap:
        b = np.sort(np.random.RandomState(seed).choice(b, cap, replace=False))
    return b

HN_COUNTS = {part: {"eligible": int(np.sum((y_train[rows] == BENIGN) & (sc[rows] >= HN_TAU))),
                    "used": int(len(hard_negatives(rows, part, SEED)))}
             for part, rows, sc in (("fit", fit_idx, P_OOF_FIT), ("full", all_idx, p_oof))}
print("benign hard negatives:", HN_COUNTS)

def select_s2r_features():
    name = f"features_s2r_{HN_TAG}.json"
    if have(name):
        return load_json(ckpt(name))
    att = fit_idx[y_train[fit_idx] != BENIGN]
    groups = [att[y_train[att] == c] for c in ATTACK_CODES] + [hard_negatives(fit_idx, "fit", SEED)]
    bi = balanced_rows(groups, np.random.RandomState(SEED))
    mi = mi_scores(X_train.iloc[bi], y_train[bi])
    kept = greedy_prune(X_train.iloc[bi], mi.head(PS_TOP_K).index.tolist(), PS_CORR)
    tab = pd.DataFrame({"feature": mi.index, "mi_stage2_with_benign": mi.values})
    tab["selected_per_stage_r"] = tab.feature.isin(kept)
    tab["selected_published"] = tab.feature.isin(F_PUB)
    tab["selected_per_stage"] = tab.feature.isin(FEATS["per_stage"])
    tab.to_csv(ckpt(f"table_feature_selection_s2r_{HN_TAG}.csv"), index=False)
    save_json(kept, ckpt(name))
    return kept

FEATS["per_stage_r"] = select_s2r_features()
if os.path.exists(ckpt(f"table_feature_selection_s2r_{HN_TAG}.csv")):
    shutil.copy(ckpt(f"table_feature_selection_s2r_{HN_TAG}.csv"), res("table_feature_selection_stage2_benign.csv"))
GEN_COLS4 = list(dict.fromkeys(F_PUB + FEATS["per_stage"] + FEATS["per_stage_r"]))   # v4 generator columns
print(f"v4 Stage-2 features: {len(FEATS['per_stage_r'])} "
      f"({len(set(FEATS['per_stage_r']) - set(FEATS['per_stage']))} not in the v3 per-stage set); "
      f"v4 generator columns: {len(GEN_COLS4)}")

## 7. Augmentation: SMOTE and generators
* **SMOTE**: interpolates between a flow and one of its 5 nearest same-class neighbours (published setting).
* **CTGAN** (Xu et al., 2019): one conditional GAN for all augmented classes, sampled conditionally on the label.
* **Per-class CTGAN** (v4): one CTGAN per augmented class (as stacked CTGAN in A-NIDS); each model gets at least
  `CTGAN_PC_MIN_STEPS` generator updates, so classes with a few hundred flows are not under-trained.
* **TVAE** (Xu et al., 2019): trained on a class-balanced table; output filtered by label.
* **CopulaGAN** (SDV): v3 reference only.

Generators are trained on the fit part, on the augmented classes only. Integer columns with at most 10 values are
modelled as categorical. **v4 post-processing is class-wise**: each synthetic flow is rounded (integer columns of
its class) and clipped to the range of its own class; columns constant within the class are set to that constant.
v3 pools (global range) are kept unchanged for the v3 systems.

In [ ]:
def generator_table(rows, cols):
    sel = rows[np.isin(y_train[rows], AUG_CODES)]
    t = X_train.iloc[sel][cols].reset_index(drop=True).copy()
    t[LABEL_COL] = [CLASS_NAMES[c] for c in y_train[sel]]
    return t

def column_info(table, cols):
    use = [c for c in cols if table[c].nunique() > 1]
    integer = [c for c in cols if np.all(np.mod(table[c].values, 1) == 0)]
    discrete = [c for c in use if c in integer and table[c].nunique() <= DISCRETE_MAX_UNIQUE]
    return {"cols": list(cols), "use": use, "const": {c: float(table[c].iloc[0]) for c in cols if c not in use},
            "integer": integer, "discrete": discrete,
            "min": table[cols].min().to_dict(), "max": table[cols].max().to_dict()}

def fit_generator(method, table, cols, seed):
    """One model for all augmented classes (label column included)."""
    info = column_info(table, cols)
    data = table[info["use"] + [LABEL_COL]].copy()
    for c in info["discrete"]:
        data[c] = data[c].astype(int).astype(str)
    disc = [LABEL_COL] + info["discrete"]
    if method == "ctgan":
        from ctgan import CTGAN
        model = CTGAN(**HP["ctgan"], verbose=False)
        model.set_random_state(seed)
        model.fit(data, discrete_columns=disc)
    elif method == "tvae":
        from ctgan import TVAE
        cap = int(min(TVAE_BALANCE_CAP, data[LABEL_COL].value_counts().max()))
        bal = pd.concat([g.sample(cap, replace=len(g) < cap, random_state=seed) for _, g in data.groupby(LABEL_COL)],
                        ignore_index=True)
        model = TVAE(**HP["tvae"])
        model.set_random_state(seed)
        model.fit(bal, discrete_columns=disc)
    elif method == "copulagan":
        import torch
        from sdv.metadata import Metadata
        from sdv.single_table import CopulaGANSynthesizer
        md = Metadata.detect_from_dataframe(data, table_name="flows", infer_keys=None)
        for c in data.columns:
            md.update_column(c, table_name="flows", sdtype="categorical" if c in disc else "numerical")
        np.random.seed(seed); torch.manual_seed(seed)
        model = CopulaGANSynthesizer(md, **HP["copulagan"])
        model.fit(data)
    else:
        raise ValueError(method)
    return model, info

def fit_class_ctgan(table_c, seed):
    """Per-class CTGAN: no label column; discrete columns, rounding and range taken from this class only."""
    from ctgan import CTGAN
    info = column_info(table_c, GEN_COLS4)
    if not info["use"]:
        return None, info
    data = table_c[info["use"]].copy()
    for c in info["discrete"]:
        data[c] = data[c].astype(int).astype(str)
    hp = dict(HP["ctgan_pc"])
    steps = max(len(data) // hp["batch_size"], 1)
    hp["epochs"] = int(max(hp["epochs"], np.ceil(CTGAN_PC_MIN_STEPS / steps)))
    model = CTGAN(**hp, verbose=False)
    model.set_random_state(seed)
    model.fit(data, discrete_columns=info["discrete"])
    return model, info

def postprocess(df, info):
    out = pd.DataFrame(index=df.index)
    for c in info["cols"]:
        if c in info["const"] or c not in df.columns:
            out[c] = info["const"].get(c, info["min"][c])
            continue
        v = pd.to_numeric(df[c], errors="coerce").astype(float)
        v = v.fillna(v.median() if v.notna().any() else info["min"][c])
        if c in info["integer"]:
            v = v.round()
        out[c] = v.clip(info["min"][c], info["max"][c])
    return out

def draw(method, model, cls_name, n, seed):
    if method == "ctgan":
        return model.sample(n, condition_column=LABEL_COL, condition_value=cls_name)
    if method == "tvae":
        return model.sample(n)
    from sdv.sampling import Condition
    import torch
    np.random.seed(seed); torch.manual_seed(seed)
    return model.sample_from_conditions([Condition(column_values={LABEL_COL: cls_name}, num_rows=n)])

def draw_class(method, model, c, need, seed):
    """Sample until `need` flows carry label c (TVAE samples all classes and is filtered)."""
    got, acc, rounds, rate_ = [], 0, 0, 1.0
    while acc < need and rounds < 25:
        batch = int(min(500_000, max(2_000, (need - acc) / max(rate_, 0.01) * 1.2)))
        d = draw(method, model, CLASS_NAMES[c], batch, seed + rounds)
        keep = d[d[LABEL_COL].astype(str) == CLASS_NAMES[c]]
        rate_ = max(len(keep) / max(len(d), 1), 1e-4)
        if len(keep):
            got.append(keep); acc += len(keep)
        rounds += 1
    return (pd.concat(got, ignore_index=True).head(need) if got else None), rate_

def need_for(c): return AUG_TARGET - int(np.sum(y_train[fit_idx] == c))

def generate_pool(method, seed, timing_only=False):
    """v3 pool: one model, global rounding and clipping (kept so v3 systems are reproduced exactly)."""
    name = f"pool_{method}_s{seed}"
    if not timing_only and have(name + ".parquet", name + ".json"):
        return pd.read_parquet(ckpt(name + ".parquet")), load_json(ckpt(name + ".json"))
    table = generator_table(fit_idx, GEN_COLS)
    with Timer() as t_fit:
        model, info = fit_generator(method, table, GEN_COLS, seed)
    parts, log = [], {"method": method, "seed": seed, "train_rows": int(len(table)), "classes": {}}
    with Timer() as t_samp:
        for c in AUG_CODES:
            if need_for(c) <= 0:
                continue
            d, r = draw_class(method, model, c, need_for(c), seed)
            if d is None:
                log["classes"][CLASS_NAMES[c]] = {"needed": need_for(c), "generated": 0}
                continue
            g = postprocess(d, info); g["_y"] = TO_COMPACT[c]; parts.append(g)
            log["classes"][CLASS_NAMES[c]] = {"needed": need_for(c), "generated": int(len(g)), "acceptance_rate": r}
    pool = pd.concat(parts, ignore_index=True)
    log.update({"fit_seconds": t_fit.s, "sample_seconds": t_samp.s, **MACHINE})
    if timing_only:
        return None, log
    pool.to_parquet(ckpt(name + ".parquet")); save_json(log, ckpt(name + ".json"))
    print(f"[{name}] generator {t_fit.s / 60:.1f} min, {len(pool):,} rows")
    return pool, log

def generate_pool4(method, seed, timing_only=False):
    """v4 pool: class-wise post-processing; 'ctgan_pc' trains one CTGAN per class."""
    name = f"pool4_{method}_s{seed}"
    if not timing_only and have(name + ".parquet", name + ".json"):
        return pd.read_parquet(ckpt(name + ".parquet")), load_json(ckpt(name + ".json"))
    table = generator_table(fit_idx, GEN_COLS4)
    tabs = {c: table[table[LABEL_COL] == CLASS_NAMES[c]].reset_index(drop=True) for c in AUG_CODES}
    cinfo = {c: column_info(tabs[c], GEN_COLS4) for c in AUG_CODES}
    parts, log = [], {"method": method, "seed": seed, "train_rows": int(len(table)), "classes": {}}
    fit_s = samp_s = 0.0
    if method != "ctgan_pc":
        with Timer() as t:
            model, _ = fit_generator(method, table, GEN_COLS4, seed)
        fit_s += t.s
    for c in AUG_CODES:
        need = need_for(c)
        if need <= 0:
            continue
        if method == "ctgan_pc":
            with Timer() as t:
                model_c, info_c = fit_class_ctgan(tabs[c], seed + int(c))
            fit_s += t.s
            with Timer() as t:
                d = model_c.sample(need) if model_c is not None else pd.DataFrame(index=range(need))
            samp_s += t.s; r = 1.0
        else:
            info_c = cinfo[c]
            with Timer() as t:
                d, r = draw_class(method, model, c, need, seed)
            samp_s += t.s
            if d is None:
                log["classes"][CLASS_NAMES[c]] = {"needed": need, "generated": 0}
                continue
        g = postprocess(d.reset_index(drop=True), info_c)
        g["_y"] = TO_COMPACT[c]; parts.append(g)
        log["classes"][CLASS_NAMES[c]] = {"needed": need, "generated": int(len(g)), "acceptance_rate": r,
                                         "real_flows": int(len(tabs[c]))}
    pool = pd.concat(parts, ignore_index=True)
    log.update({"fit_seconds": fit_s, "sample_seconds": samp_s, **MACHINE})
    if timing_only:
        return None, log
    pool.to_parquet(ckpt(name + ".parquet")); save_json(log, ckpt(name + ".json"))
    print(f"[{name}] generator {fit_s / 60:.1f} min, sampling {samp_s / 60:.1f} min, {len(pool):,} rows")
    return pool, log

## 8. Synthetic data: generation and quality
Quality per augmented class in the published feature space, real fit-part flows against up to 2,000 synthetic flows:
mean Kolmogorov–Smirnov statistic (0 = identical marginals), mean absolute correlation difference, classifier
two-sample test AUC (LightGBM, 3-fold; 0.5 = indistinguishable), median distance from a synthetic flow to its closest
real flow (DCR) next to the real-to-real median, and the share of exact copies. `ctgan-v3` is the v3 pool (global
clipping), so the effect of class-wise post-processing can be read directly.

In [ ]:
GEN_LOG = {m: generate_pool4(m, SEED)[1] for m in R_METHODS if m in GAN_METHODS}

def smote_synthetic(seed):
    att = fit_idx[y_train[fit_idx] != BENIGN]
    X = X_train.iloc[att][F_PUB].reset_index(drop=True); yc = TO_COMPACT[y_train[att]]
    cnt = np.bincount(yc, minlength=A)
    strat = {int(c): AUG_TARGET for c in AUG_COMPACT if 0 < cnt[c] < AUG_TARGET}
    k = max(1, min(SMOTE_K, int(min(cnt[list(strat)])) - 1))
    Xr, yr = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed).fit_resample(X, yc)
    syn = pd.DataFrame(np.asarray(Xr)[len(X):], columns=F_PUB); syn["_y"] = np.asarray(yr)[len(X):]
    return syn

def quality_row(real, syn, method, cls):
    rng = np.random.RandomState(SEED)
    syn = syn.iloc[rng.permutation(len(syn))[:QUALITY_SYN_ROWS]] if len(syn) > QUALITY_SYN_ROWS else syn
    ks = np.mean([ks_2samp(real[c], syn[c]).statistic for c in F_PUB])
    cr, cs = real[F_PUB].corr().fillna(0).values, syn[F_PUB].corr().fillna(0).values
    off = ~np.eye(len(F_PUB), dtype=bool)
    corr_diff = float(np.abs(cr - cs)[off].mean())
    n = int(min(len(real), len(syn), QUALITY_SYN_ROWS))
    rr = real.iloc[rng.permutation(len(real))[:n]][F_PUB].values; ss = syn.iloc[:n][F_PUB].values
    Xc = np.vstack([rr, ss]); yc2 = np.r_[np.zeros(n), np.ones(n)]
    c2st = np.nan
    if n >= 10:
        oof = cross_val_predict(LGBMClassifier(n_estimators=100, num_leaves=15, verbose=-1, random_state=SEED, n_jobs=-1),
                                Xc, yc2, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                                method="predict_proba")[:, 1]
        c2st = roc_auc_score(yc2, oof)
    mu, sd = real[F_PUB].mean().values, real[F_PUB].std().replace(0, 1).values
    R, S = (real[F_PUB].values - mu) / sd, (syn[F_PUB].values - mu) / sd
    nn = NearestNeighbors(n_neighbors=2).fit(R)
    d_syn = nn.kneighbors(S, n_neighbors=1)[0][:, 0]
    d_real = nn.kneighbors(R, n_neighbors=2)[0][:, 1]
    return {"method": method, "class": cls, "real_flows": int(len(real)), "synthetic_flows_checked": int(len(syn)),
            "ks_mean": ks, "corr_diff": corr_diff, "c2st_auc": c2st, "dcr_synthetic_median": float(np.median(d_syn)),
            "nn_real_median": float(np.median(d_real)), "exact_copy_rate": float((d_syn < 1e-9).mean())}

QNAME = "quality4_" + "_".join(sorted(GEN_LOG)) + ".json"
if have(QNAME):
    QUALITY = pd.DataFrame(load_json(ckpt(QNAME)))
else:
    sources = {"smote": smote_synthetic(SEED)}
    for m in GEN_LOG:
        sources[m] = generate_pool4(m, SEED)[0]
    if have("pool_ctgan_s42.parquet"):
        sources["ctgan-v3"] = pd.read_parquet(ckpt("pool_ctgan_s42.parquet"))
    rows = []
    for m, syn_all in sources.items():
        for c in AUG_CODES:
            real = X_train.iloc[fit_idx[y_train[fit_idx] == c]][F_PUB].reset_index(drop=True)
            syn = syn_all[syn_all._y == TO_COMPACT[c]][F_PUB].reset_index(drop=True)
            if len(syn):
                rows.append(quality_row(real, syn, m, CLASS_NAMES[c]))
    QUALITY = pd.DataFrame(rows)
    save_json(QUALITY.to_dict(orient="list"), ckpt(QNAME))
QUALITY.to_csv(res("table_synthetic_quality.csv"), index=False)
QSUM = QUALITY.groupby("method")[["ks_mean", "corr_diff", "c2st_auc", "dcr_synthetic_median", "nn_real_median",
                                  "exact_copy_rate"]].mean()
QSUM.to_csv(res("table_synthetic_quality_summary.csv"))
print(QSUM.round(3).to_string())
pd.DataFrame([{"method": m, **{k: v for k, v in GEN_LOG[m].items() if k in ("train_rows", "fit_seconds", "sample_seconds")}}
              for m in GEN_LOG]).to_csv(res("table_generator_timing.csv"), index=False)

## 9. Stage 2
Every Stage-2 variant is a spec: augmentation method, feature set, benign class yes/no, pool version and, for the
benign-class variants, the hard-negative setting (score threshold, cap). v3 and v4 variants load their checkpoints.
With a benign class, Stage 2 has A + 1 outputs and the decision becomes

> attack type = argmax of Stage 2 if p₁ ≥ τ and that argmax is an attack class; benign otherwise.

Composed probabilities (for ROC-AUC / PR-AUC): P(benign) = 1 − p₁ + p₁·p₂(benign), P(c) = p₁·p₂(c).

In [ ]:
def spec(method, feat, reject, pool="v4", hn=None):
    return {"method": method, "feat": feat, "reject": reject, "pool": pool, "hn": tuple(hn) if hn else HN_MAIN}

def s2_name(sp, part, seed):
    m, f = sp["method"], sp["feat"]
    if not sp["reject"] and (m in ("none", "smote") or sp["pool"] == "v3"):
        return f"s2_{m}_{f}_{part}_s{seed}"                                   # v3 checkpoint name
    if sp["reject"] and sp["hn"] != HN_MAIN:     # item 3: other hard-negative setting, Stage-2 features of the main one
        return f"s2v5_{m}_{f}_r_{hn_tag(sp['hn'])}_feat_{HN_TAG}_{part}_s{seed}"
    return f"s2v4_{m}_{f}_" + (f"r_{hn_tag(sp['hn'])}" if sp["reject"] else "a") + f"_{part}_s{seed}"

def stage2_data(rows, sp, seed, part):
    method, cols = sp["method"], FEATS[sp["feat"]]
    att = rows[y_train[rows] != BENIGN]
    X = X_train.iloc[att][cols].reset_index(drop=True)
    yc = TO_COMPACT[y_train[att]]
    n_real = len(yc)
    if method == "smote":
        cnt = np.bincount(yc, minlength=A)
        strat = {int(c): AUG_TARGET for c in AUG_COMPACT if 0 < cnt[c] < AUG_TARGET}
        if strat:
            k = max(1, min(SMOTE_K, int(min(cnt[list(strat)])) - 1))
            Xr, yr = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed).fit_resample(X, yc)
            X, yc = pd.DataFrame(np.asarray(Xr), columns=cols), np.asarray(yr)
    elif method in GAN_METHODS:
        pool = (generate_pool if sp["pool"] == "v3" else generate_pool4)(method, seed)[0]
        add = [pool[pool._y == c].head(AUG_TARGET - int(np.sum(yc == c))) for c in AUG_COMPACT
               if AUG_TARGET - int(np.sum(yc == c)) > 0]
        if add:
            add = pd.concat(add, ignore_index=True)
            X = pd.concat([X, add[cols]], ignore_index=True)
            yc = np.concatenate([yc, add._y.values.astype(int)])
    n_syn, n_hn = len(yc) - n_real, 0
    if sp["reject"]:
        hn = hard_negatives(rows, part, seed, sp["hn"])
        if len(hn):
            X = pd.concat([X, X_train.iloc[hn][cols].reset_index(drop=True)], ignore_index=True)
            yc = np.concatenate([yc, np.full(len(hn), A)])
        n_hn = int(len(hn))
    return X, yc, n_syn, n_hn

def stage2(sp, part, seed=SEED):
    name = s2_name(sp, part, seed)
    if have(name + ".npy", name + ".json"):
        return np.load(ckpt(name + ".npy")), load_json(ckpt(name + ".json"))
    rows, X_eval = (fit_idx, X_train.iloc[val_idx]) if part == "fit" else (all_idx, X_test)
    with Timer() as t_data:
        X, yc, n_syn, n_hn = stage2_data(rows, sp, seed, part)
    stack = build_stack(int(yc.max()) + 1, seed)
    with Timer() as t_fit:
        stack.fit(X, yc)
    with Timer() as t_pred:
        cond = stack.predict_proba(X_eval[FEATS[sp["feat"]]]).astype(np.float32)
    info = {"train_rows": int(len(yc)), "synthetic_rows": int(n_syn), "hard_negative_rows": n_hn,
            "data_seconds": t_data.s, "train_seconds": t_fit.s, "predict_seconds": t_pred.s, **MACHINE}
    np.save(ckpt(name + ".npy"), cond); save_json(info, ckpt(name + ".json"))
    if SAVE_MODELS and part == "full" and seed == SEED: joblib.dump(stack, ckpt(name + ".joblib"))
    print(f"[{name}] {t_fit.s / 60:.1f} min, {n_syn:,} synthetic, {n_hn:,} benign hard negatives")
    return cond, info

def codes(cond): return R_CODES if cond.shape[1] == A + 1 else ATTACK_CODES
def decide(p_att, cond, tau): return np.where(p_att >= tau, codes(cond)[cond.argmax(1)], BENIGN)

def composed_proba(p_att, cond):
    proba = np.zeros((len(p_att), K), dtype=np.float32)
    proba[:, BENIGN] = 1 - p_att
    proba[:, ATTACK_CODES] = p_att[:, None] * cond[:, :A]
    if cond.shape[1] == A + 1:
        proba[:, BENIGN] += p_att * cond[:, A]
    return proba

VARIANTS = {}
if RUN_V3_SYSTEMS:
    for m in AUG_METHODS_V3:
        VARIANTS[f"T-{mcode(m)}"] = spec(m, "published", False, "v3")
    VARIANTS["T-CTGAN-PS"] = spec("ctgan", "per_stage", False, "v3")
if RUN_CHAIN:
    VARIANTS["T-CTGAN-PS-CC"] = spec("ctgan", "per_stage", False, "v4")
for m in R_METHODS:
    VARIANTS[f"R-{mcode(m)}"] = spec(m, "per_stage_r", True, "v4")
NEED_VAL = [k for k in VARIANTS if k.startswith("R-")] + \
           ([f"T-{mcode(m)}" for m in VAL_METHODS_V3] if RUN_V3_SYSTEMS else [])

COND_VAL, COND_TEST, S2_INFO = {}, {}, {}
for k in NEED_VAL:
    COND_VAL[k], _ = stage2(VARIANTS[k], "fit")
for k, sp in VARIANTS.items():
    COND_TEST[k], S2_INFO[k] = stage2(sp, "full")

# Stage 2 on true attack flows only (benign-class variants may send an attack flow to benign: counted as an error)
iso = []
for k, cond in COND_TEST.items():
    yt, yp = TO_COMPACT[y_test[tatt]], cond[tatt].argmax(1)
    p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=np.arange(A), zero_division=0)
    iso.append({"variant": k, "features": VARIANTS[k]["feat"], "benign_class": VARIANTS[k]["reject"],
                "macro_f1": f.mean(), "macro_precision": p.mean(), "macro_recall": r.mean(),
                "augmented_classes_macro_f1": f[AUG_COMPACT].mean(), "attacks_sent_to_benign": float((yp == A).mean()),
                **{f"f1_{ATTACK_NAMES[i]}": f[i] for i in range(A)}})
ISO = pd.DataFrame(iso); ISO.to_csv(res("table_stage2_true_attacks.csv"), index=False)
print(ISO[["variant", "macro_f1", "augmented_classes_macro_f1", "attacks_sent_to_benign"]].round(4).to_string(index=False))

## 10. Threshold rule and proposed system (items 2 and 4)
**Rule.** For a false-alarm budget b, τ is the value with the highest validation macro-F1 among those whose
validation benign false-alarm rate (final decision, after Stage 2) is at most `FPR_MARGIN` × b. In v4 the benign-class
systems on CSE-CIC-IDS2018 returned more gate false alarms to benign on validation than on test, so the test rate went
over the 1 % budget; the margin leaves room for that gap. If no τ meets the target, the τ with the fewest validation
false alarms is used and the row is flagged (`target_met_on_validation = False`). τ is searched on `TAU_GRID_VAL`;
benign-class systems search τ at or above their own hard-negative threshold.
* Main rule: b = `MAX_BENIGN_FPR`. Section 14 applies the same rule to every budget in `FPR_CAPS`.
* The proposed system is fixed in Section 0 (`PROPOSED_GEN`); nothing is selected on test.

In [ ]:
def f1_vector(yt, yp):
    cm = np.bincount(yt * K + yp, minlength=K * K).reshape(K, K)
    denom = cm.sum(0) + cm.sum(1)
    return np.where(denom > 0, 2 * np.diag(cm) / np.maximum(denom, 1), 0.0), denom > 0

def macro_f1_fast(yt, yp):
    f, present = f1_vector(yt, yp)
    return f[present].mean()

def ftau(t): return f"{t:.3f}".rstrip("0").rstrip(".") if t is not None else "-"

def grid_for(k):
    sp = VARIANTS.get(k)
    if sp is not None and sp["reject"]:
        return TAU_GRID_VAL[TAU_GRID_VAL >= sp["hn"][0] - 1e-9]
    return TAU_GRID_VAL

def choose_tau(score, j, cap, grid):
    """score: attack score per validation flow; j: class assigned when score >= tau (BENIGN = sent back)."""
    sb, kb = score[vb], j[vb] != BENIGN
    lim = None if cap is None else FPR_MARGIN * cap
    best = low = None
    for t in grid:
        fpr = float(np.mean((sb >= t) & kb))
        row = {"tau": float(t), "val_benign_fpr": fpr, "val_score_fpr": float(np.mean(sb >= t))}
        if lim is not None and fpr > lim + 1e-12:
            if low is None or fpr < low["val_benign_fpr"]:
                low = row
            continue
        row["val_macro_f1"] = macro_f1_fast(y_val, np.where(score >= t, j, BENIGN))
        if best is None or row["val_macro_f1"] > best["val_macro_f1"]:
            best = row
    if best is None:
        low["val_macro_f1"] = macro_f1_fast(y_val, np.where(score >= low["tau"], j, BENIGN))
        best = low
    best.update({"budget": cap, "validation_target": lim,
                 "target_met_on_validation": lim is None or best["val_benign_fpr"] <= lim + 1e-12})
    return best

def two_inputs(k, split):
    cond = COND_VAL[k] if split == "val" else COND_TEST[k]
    return (p_val_used if split == "val" else p_test), codes(cond)[cond.argmax(1)]

TAU_VAL = {k: choose_tau(*two_inputs(k, "val"), MAX_BENIGN_FPR, grid_for(k)) for k in COND_VAL}
PROPOSED = f"R-{mcode(PROPOSED_GEN)}"
assert PROPOSED in TAU_VAL, PROPOSED
print("proposed system (fixed):", PROPOSED, {k: (ftau(v["tau"]), round(float(v["val_macro_f1"]), 4)) for k, v in TAU_VAL.items()})

THRESHOLDS = pd.DataFrame([{"rule": "fixed (TAU_MAIN)", "tau": TAU_MAIN},
                           {"rule": "published (OOF benign-vs-attack F1)", "tau": TAU_PUB}] +
                          [{"rule": f"validation macro-F1, final FPR <= {FPR_MARGIN:g} x {MAX_BENIGN_FPR:g}", "variant": k,
                            **TAU_VAL[k]} for k in TAU_VAL])
THRESHOLDS["validation_gate_scores"] = VAL_SRC
THRESHOLDS.to_csv(res("table_thresholds.csv"), index=False)
print(THRESHOLDS.round(4).to_string(index=False))

## 11. Single-stage baselines (item 1)
15-class models on the published features, trained without augmentation. Full models (training split → test) load
their v3/v4 checkpoints. For thresholding, the models in `S1_THRESHOLDED` are also fitted on the fit part and score the
validation split; the models in `S1_SEED_MODELS` are refitted with every seed in `SEEDS_REPEAT`.

A thresholded single-stage model flags a flow when its attack score 1 − P(benign) ≥ τ and then assigns the attack
class with the highest probability; τ follows the rule of Section 10. Without a threshold (rows S1-*), the class with
the highest probability is taken, as in v4.

In [ ]:
def to_k_columns(model, proba):
    out = np.zeros((len(proba), K), np.float32)
    out[:, np.asarray(model.classes_, int)] = proba
    return out

def flat_model(key, part="full", seed=SEED):
    """part 'full': trained on the training split, scores test; part 'fit': trained on the fit part, scores validation."""
    base = FLAT_CKPT[key] + ("_fit" if part == "fit" else "") + ("" if seed == SEED else f"_s{seed}")
    if have(base + "_proba.npy", base + ".json"):
        return np.load(ckpt(base + "_proba.npy")), load_json(ckpt(base + ".json"))
    rows, X_eval = (all_idx, X_test) if part == "full" else (fit_idx, X_train.iloc[val_idx])
    Xf = X_train[F_PUB] if part == "full" else X_train.iloc[rows][F_PUB]
    model = FLAT_BUILDERS[key](seed)
    with Timer() as t_fit:
        model.fit(Xf, y_train[rows])
    with Timer() as t_pred:
        proba = to_k_columns(model, model.predict_proba(X_eval[F_PUB]))
    info = {"part": part, "seed": seed, "train_rows": int(len(rows)), "train_seconds": t_fit.s,
            "predict_seconds": t_pred.s, "predict_rows": int(len(X_eval)), **MACHINE}
    if key == "lgbm":
        info["max_abs_raw_score"] = float(np.abs(model.predict(X_eval[F_PUB], raw_score=True)).max())
    np.save(ckpt(base + "_proba.npy"), proba); save_json(info, ckpt(base + ".json"))
    if part == "full" and seed == SEED:
        np.save(ckpt(base + "_pred.npy"), proba.argmax(1))
    del model
    print(f"[{base}] {t_fit.s / 60:.1f} min")
    return proba, info

def flat_inputs(P):
    return 1.0 - P[:, BENIGN], ATTACK_CODES[P[:, ATTACK_CODES].argmax(1)]

FLAT = {}
for key in RUN_FLAT:
    proba, info = flat_model(key)
    FLAT[key] = (proba.argmax(1), proba, info)
FLAT_VAL = {key: flat_model(key, "fit")[0] for key in S1_THRESHOLDED if key in FLAT}
FLAT_SEED = {(key, s): (FLAT[key][1] if s == SEED else flat_model(key, "full", s)[0])
             for key in S1_SEED_MODELS if key in FLAT for s in SEEDS_REPEAT}
print("single-stage models:", list(FLAT), "| fit-part models:", list(FLAT_VAL),
      "| seed models:", sorted({k for k, _ in FLAT_SEED}))

## 12. Published methods under this protocol (item 6)
Two rebalancing methods from the CSE-CIC-IDS2018 literature are re-implemented and run on the same split, cleaning,
classes and metrics as every other system. Both use all numeric flow features and take the class with the highest
probability.
* **SMOTE + RF / DT** (Karatas et al., IEEE Access, 2020): SMOTE (5 neighbours) raises every attack class with fewer
  than (largest class) / `KARATAS_RATIO` training flows to that size; random forest (100 trees) and decision tree with
  scikit-learn defaults. The paper merged the attacks into 6 classes and used 5-fold cross-validation; here all 15
  classes and the common split are used.
* **DSSTE + RF / XGBoost** (Liu et al., IEEE Access, 2021): on a class-capped subsample of the training split (the
  paper used a 100,000-flow subset), edited nearest neighbours (ENN) splits the flows into an easy set and a difficult
  set; difficult flows of majority classes are replaced by k-means centres (`compress` × their number); difficult flows
  of minority classes are kept and copied `zoom_k` times with continuous columns multiplied by a random factor in
  [1 − 1/k, 1 + 1/k] (discrete columns unchanged). Random forest (100 trees) and XGBoost (default settings) are trained
  on the result. Majority / minority follows `MINORITY_CUTOFF`. **Check `DSSTE_PARAMS` against the paper.**

Settings and the differences from the papers are written to `table_published_settings.csv`.

In [ ]:
ALL_FEATS = list(X_train.columns)
PUB = {}

def karatas_train(seed=SEED):
    X, y = X_train[ALL_FEATS].to_numpy(np.float32), y_train
    target = int(np.ceil(train_counts.max() / KARATAS_RATIO))
    strat = {int(c): target for c in ATTACK_CODES if 1 < train_counts[c] < target}
    if not strat:
        return X, y, {"smote_target": target, "synthetic_rows": 0}
    k = max(1, min(SMOTE_K, int(min(train_counts[c] for c in strat)) - 1))
    Xr, yr = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=seed).fit_resample(X, y)
    return np.asarray(Xr, np.float32), np.asarray(yr), {"smote_target": target, "smote_k": k,
                                                         "classes_raised": len(strat), "synthetic_rows": int(len(yr) - len(y))}

def dsste_train(seed=SEED):
    P, rng = DSSTE_PARAMS, np.random.RandomState(seed)
    rows = []
    for c in range(K):
        idx = np.flatnonzero(y_train == c)
        cap = P["train_cap_benign"] if c == BENIGN else P["train_cap_attack"]
        rows.append(idx if len(idx) <= cap else rng.choice(idx, cap, replace=False))
    rows = np.sort(np.concatenate(rows))
    X, y = X_train.iloc[rows][ALL_FEATS].to_numpy(np.float64), y_train[rows]
    sc = StandardScaler().fit(X)
    Z = sc.transform(X)
    k = P["enn_k"]
    nb = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(Z).kneighbors(return_distance=False)   # self excluded
    agree = y[nb] == y[:, None]
    difficult = ~agree.all(1) if P["enn_kind"] == "all" else (2 * agree.sum(1) <= k)
    majority = train_counts >= MINORITY_CUTOFF
    integer = np.all(np.mod(X, 1) == 0, axis=0)
    nuniq = np.array([len(np.unique(X[:, i])) for i in range(X.shape[1])])
    cont = ~(integer & (nuniq <= DISCRETE_MAX_UNIQUE))
    lo, hi = 1 - 1 / P["zoom_k"], 1 + 1 / P["zoom_k"]
    parts_X, parts_y, log = [X[~difficult]], [y[~difficult]], {}
    for c in np.unique(y):
        m = difficult & (y == c); n = int(m.sum())
        if n == 0:
            continue
        if majority[c]:
            n_cl = max(1, int(round(P["compress"] * n)))
            if n_cl < n:
                km = MiniBatchKMeans(n_clusters=n_cl, random_state=seed, n_init=3,
                                     batch_size=max(1024, 3 * n_cl)).fit(Z[m])
                parts_X.append(sc.inverse_transform(km.cluster_centers_)); parts_y.append(np.full(n_cl, c))
            else:
                parts_X.append(X[m]); parts_y.append(y[m])
            log[CLASS_NAMES[c]] = {"difficult": n, "kept_as_centres": min(n_cl, n)}
        else:
            Xm = X[m]; parts_X.append(Xm); parts_y.append(y[m])
            for _ in range(P["zoom_k"]):
                Xz = Xm.copy(); Xz[:, cont] *= rng.uniform(lo, hi, size=(n, int(cont.sum())))
                parts_X.append(Xz); parts_y.append(np.full(n, c))
            log[CLASS_NAMES[c]] = {"difficult": n, "zoomed_copies": n * P["zoom_k"]}
    Xd, yd = np.vstack(parts_X).astype(np.float32), np.concatenate(parts_y)
    return Xd, yd, {"subsample_rows": int(len(rows)), "difficult_rows": int(difficult.sum()),
                    "easy_rows": int((~difficult).sum()), "continuous_columns": int(cont.sum()),
                    "train_rows_after_dsste": int(len(yd)), "classes": log}

def pub_model(code, n_classes, seed=SEED):
    if code.endswith("-RF"):
        return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
    if code.endswith("-DT"):
        return DecisionTreeClassifier(random_state=seed)
    if code.endswith("-XGB"):
        return XGBClassifier(objective="multi:softprob", num_class=n_classes, tree_method="hist", n_jobs=-1,
                             random_state=seed, eval_metric="mlogloss")
    raise ValueError(code)

def run_published(code):
    name = f"pub_{code.lower()}"
    if have(name + "_proba.npy", name + ".json"):
        return np.load(ckpt(name + "_proba.npy")), load_json(ckpt(name + ".json"))
    fam = code.split("-")[0]
    if PUB_DATA.get(fam) is None:
        with Timer() as t_data:
            PUB_DATA[fam] = (karatas_train if fam == "KARATAS" else dsste_train)(SEED)
        PUB_DATA[fam] = (*PUB_DATA[fam], t_data.s)
    X, y, dlog, data_s = PUB_DATA[fam]
    present = np.unique(y); enc = np.searchsorted(present, y)
    model = pub_model(code, len(present))
    with Timer() as t_fit:
        model.fit(X, enc)
    with Timer() as t_pred:
        Pt = model.predict_proba(X_TEST_ALL)
    proba = np.zeros((len(Pt), K), np.float32); proba[:, present] = Pt
    info = {"code": code, "train_rows": int(len(y)), "features": len(ALL_FEATS), "data_seconds": data_s,
            "train_seconds": t_fit.s, "predict_seconds": t_pred.s, "data_log": dlog, **MACHINE}
    np.save(ckpt(name + "_proba.npy"), proba); save_json(info, ckpt(name + ".json"))
    print(f"[{name}] data {data_s / 60:.1f} min, fit {t_fit.s / 60:.1f} min, {len(y):,} training rows")
    return proba, info

PUB_DATA = {}
if RUN_PUBLISHED:
    X_TEST_ALL = X_test[ALL_FEATS].to_numpy(np.float32)
    for code in PUBLISHED:
        proba, info = run_published(code)
        PUB[code] = (proba.argmax(1), proba, info)
    PUB_DATA.clear(); del X_TEST_ALL
    PUB_SETTINGS = pd.DataFrame([
        {"code": "KARATAS-RF / KARATAS-DT", "paper": "Karatas et al., IEEE Access 2020",
         "rebalancing": f"SMOTE (k = {SMOTE_K}) on the training split; attack classes below (largest class)/{KARATAS_RATIO} "
                        "raised to that size",
         "classifiers": "random forest (100 trees), decision tree; scikit-learn defaults",
         "features": f"all {len(ALL_FEATS)} numeric features",
         "differences_from_paper": "15 classes instead of 6 merged classes; common 80/20 split instead of 5-fold CV; "
                                   "rows with NaN/inf dropped instead of replaced; KNN, GB, AdaBoost and LDA not run"},
        {"code": "DSSTE-RF / DSSTE-XGB", "paper": "Liu et al., IEEE Access 2021",
         "rebalancing": f"DSSTE on a subsample (benign <= {DSSTE_PARAMS['train_cap_benign']:,}, each attack class <= "
                        f"{DSSTE_PARAMS['train_cap_attack']:,}); ENN k = {DSSTE_PARAMS['enn_k']} ({DSSTE_PARAMS['enn_kind']}); "
                        f"k-means compression {DSSTE_PARAMS['compress']:g}; zoom k = {DSSTE_PARAMS['zoom_k']}",
         "classifiers": "random forest (100 trees), XGBoost (default settings, hist)",
         "features": f"all {len(ALL_FEATS)} numeric features",
         "differences_from_paper": "15 classes instead of 6; evaluated on the full test split; SVM, LSTM, AlexNet and "
                                   "Mini-VGGNet not run; parameters to be checked against the paper"}])
    PUB_SETTINGS.to_csv(res("table_published_settings.csv"), index=False)
    save_json({c: PUB[c][2] for c in PUB}, res("published_methods_log.json"))

## 13. Test results
One row per system. Training and inference times are not in this table until Section 20, which measures them on
this machine.

In [ ]:
def full_metrics(y_true, y_pred, proba):
    labels = np.arange(K)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    is_b, pred_b = y_true == BENIGN, y_pred == BENIGN
    keep = [c for c in range(K) if c != INFIL]
    m = {"macro_f1": f.mean(), "minority_macro_f1": f[MINORITY_FULL].mean(), "macro_f1_excl_infiltration": f[keep].mean(),
         "infiltration_f1": f[INFIL] if INFIL is not None else np.nan,
         "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
         "accuracy": accuracy_score(y_true, y_pred), "mcc": matthews_corrcoef(y_true, y_pred),
         "macro_precision": p.mean(), "macro_recall": r.mean(),
         "attack_detection_rate": float((~pred_b[~is_b]).mean()), "benign_fpr": float((~pred_b[is_b]).mean()),
         "benign_flows_flagged": int((~pred_b[is_b]).sum())}
    if proba is not None:
        P = np.asarray(proba, np.float64); P /= np.maximum(P.sum(1, keepdims=True), 1e-12)
        m["macro_roc_auc"] = roc_auc_score(y_true, P, multi_class="ovr", average="macro", labels=labels)
        m["macro_pr_auc"] = average_precision_score(label_binarize(y_true, classes=labels), P, average="macro")
    pc = pd.DataFrame({"class": CLASS_NAMES, "precision": p, "recall": r, "f1": f, "support": s})
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=CLASS_NAMES, columns=CLASS_NAMES)
    return m, pc, cm

NAMES = {"none": "no augmentation", "smote": "SMOTE", "ctgan": "CTGAN", "ctgan_pc": "per-class CTGAN",
         "tvae": "TVAE", "copulagan": "CopulaGAN"}
PUB_LABELS = {"KARATAS-RF": "SMOTE + random forest (Karatas et al., re-run)",
              "KARATAS-DT": "SMOTE + decision tree (Karatas et al., re-run)",
              "DSSTE-RF": "DSSTE + random forest (Liu et al., re-run)",
              "DSSTE-XGB": "DSSTE + XGBoost (Liu et al., re-run)"}
SYSTEMS = []   # (code, label, role, kind, key, tau)
for key in RUN_FLAT:
    SYSTEMS.append((f"S1-{key.upper()}", FLAT_LABELS[key], "single-stage baseline", "flat", key, None))
for code in PUB:
    SYSTEMS.append((code, PUB_LABELS.get(code, code), "published method (re-run)", "pub", code, None))
if RUN_V3_SYSTEMS:
    for m in AUG_METHODS_V3:
        SYSTEMS.append((f"T-{mcode(m)}", f"v3 two-stage + {NAMES[m]}, τ = {TAU_MAIN:.2f}", "v3 reference", "two",
                        f"T-{mcode(m)}", TAU_MAIN))
    if "smote" in AUG_METHODS_V3:
        SYSTEMS.append(("T-SMOTE-PUB", f"v3 two-stage + SMOTE, τ = {TAU_PUB:.2f} (published rule)",
                        "first-paper configuration", "two", "T-SMOTE", TAU_PUB))
    SYSTEMS.append(("T-CTGAN-PS", f"v3 + Stage-2 features from attack flows, τ = {TAU_MAIN:.2f}", "ablation step",
                    "two", "T-CTGAN-PS", TAU_MAIN))
if RUN_CHAIN:
    SYSTEMS.append(("T-CTGAN-PS-CC", f"+ class-wise post-processing, τ = {TAU_MAIN:.2f}", "ablation step", "two",
                    "T-CTGAN-PS-CC", TAU_MAIN))
for m in R_METHODS:
    k = f"R-{mcode(m)}"
    SYSTEMS.append((k, f"Benign-class Stage 2 + {NAMES[m]}, τ = {ftau(TAU_VAL[k]['tau'])} (validation)",
                    "proposed" if k == PROPOSED else "augmentation comparison", "two", k, TAU_VAL[k]["tau"]))
for m in PROPOSED_CANDIDATES:
    k = f"R-{mcode(m)}"
    SYSTEMS.append((f"{k}-45", f"Benign-class Stage 2 + {NAMES[m]}, τ = {TAU_MAIN:.2f}", "threshold ablation", "two",
                    k, TAU_MAIN))
SYS_KEY = {s[0]: (s[4], s[5]) for s in SYSTEMS if s[3] == "two"}

RESULTS, ROWS = {}, []
def add_result(code, label, role, pred, proba, tau=None, extra=None):
    met, pc, cm = full_metrics(y_test, pred, proba)
    RESULTS[code] = {"pred": pred, "metrics": met, "per_class": pc, "cm": cm}
    ROWS.append({"code": code, "system": label, "role": role, "tau": tau, **met, **(extra or {})})
    cm.to_csv(res(f"confusion_matrix_{code}.csv"))

def save_table():
    global TABLE
    TABLE = pd.DataFrame(ROWS).set_index("code")
    TABLE.to_csv(res("table_results.csv"))
    return TABLE

for code, label, role, kind, key, tau in SYSTEMS:
    if kind == "flat":
        pred, proba, info = FLAT[key]
        add_result(code, label, role, pred, proba, None, {"training_rows": int(len(y_train))})
    elif kind == "pub":
        pred, proba, info = PUB[key]
        add_result(code, label, role, pred, proba, None, {"training_rows": info["train_rows"]})
    else:
        info = S2_INFO[key]
        add_result(code, label, role, decide(p_test, COND_TEST[key], tau), composed_proba(p_test, COND_TEST[key]), tau,
                   {"stage2_train_rows": info.get("train_rows"), "stage2_synthetic_rows": info.get("synthetic_rows"),
                    "stage2_hard_negatives": info.get("hard_negative_rows", 0)})
save_table()
show = ["macro_f1", "minority_macro_f1", "macro_f1_excl_infiltration", "infiltration_f1", "accuracy", "mcc",
        "attack_detection_rate", "benign_fpr"]
print(TABLE[["tau"] + show].round(4).to_string())

## 14. False-alarm budgets (items 1 and 2)
Every system in `BUDGET_SYSTEMS`, the thresholded single-stage models included, gets one τ per budget in `FPR_CAPS`
by the rule of Section 10 and is scored once on test. `test_over_budget` marks rows whose test false-alarm rate is
above the budget; `test_to_validation_fpr_ratio` shows how far validation under- or over-estimated it.

**Seeds.** τ stays at the value chosen with the seed-42 validation models; the test model is replaced by the one
trained with each seed in `SEEDS_REPEAT` (S1-STACK and the two-stage systems in `SEED_SYSTEMS`).

The systems in `BUDGET_ROWS` at `FA_BUDGET` are added to the results table as `<code>-FA1` (budget in per cent).

In [ ]:
def sys_code(c): return PROPOSED if c == "PROPOSED" else c

def budget_inputs(code, split, seed=SEED):
    if code.startswith("S1-"):
        key = code[3:].lower()
        if split == "val":
            return flat_inputs(FLAT_VAL[key])
        return flat_inputs(FLAT[key][1] if seed == SEED else FLAT_SEED[(key, seed)])
    if split == "val" or seed == SEED:
        return two_inputs(code, split)
    cond = stage2(VARIANTS[code], "full", seed=seed)[0]
    return p_test, codes(cond)[cond.argmax(1)]

def test_row(pred):
    f, _ = f1_vector(y_test, pred)
    keep = [c for c in range(K) if c != INFIL]
    return {"test_macro_f1": f.mean(), "test_minority_macro_f1": f[MINORITY_FULL].mean(),
            "test_infiltration_f1": f[INFIL] if INFIL is not None else np.nan,
            "test_macro_f1_excl_infiltration": f[keep].mean(),
            "test_benign_fpr": float((pred[tb] != BENIGN).mean()),
            "test_detection_rate": float((pred[tatt] != BENIGN).mean())}

FA_TAG = f"FA{100 * FA_BUDGET:g}"
assert FA_BUDGET in FPR_CAPS, "FA_BUDGET must be one of FPR_CAPS"
BUD_SYS = [c for c in dict.fromkeys([sys_code(c) for c in BUDGET_SYSTEMS] + [PROPOSED])
           if (c.startswith("S1-") and c[3:].lower() in FLAT_VAL) or c in COND_VAL]
bud, BUD_PRED, BUD_TAU = [], {}, {}
for code in BUD_SYS:
    sv, jv = budget_inputs(code, "val")
    st, jt = budget_inputs(code, "test")
    for cap in FPR_CAPS:
        b = choose_tau(sv, jv, cap, grid_for(code))
        pred = np.where(st >= b["tau"], jt, BENIGN)
        r = {"system": code, **b, **test_row(pred)}
        r["test_over_budget"] = r["test_benign_fpr"] > cap
        r["test_to_validation_fpr_ratio"] = r["test_benign_fpr"] / max(r["val_benign_fpr"], 1e-12)
        bud.append(r); BUD_PRED[(code, cap)] = pred; BUD_TAU[(code, cap)] = b["tau"]
BUDGET = pd.DataFrame(bud)
BUDGET = BUDGET[["system", "budget", "validation_target"] + [c for c in BUDGET.columns
                                                             if c not in ("system", "budget", "validation_target")]]
BUDGET.to_csv(res("table_budget_comparison.csv"), index=False)
print(BUDGET[["system", "budget", "tau", "target_met_on_validation", "val_benign_fpr", "test_benign_fpr",
              "test_macro_f1", "test_minority_macro_f1", "test_infiltration_f1", "test_detection_rate"]]
      .round(4).to_string(index=False))

seed_codes = list(dict.fromkeys(c for c in (sys_code(s) for s in SEED_SYSTEMS) if c in SYS_KEY))
BUD_SEED_SYS = [c for c in dict.fromkeys([f"S1-{k.upper()}" for k in S1_SEED_MODELS] + seed_codes) if c in BUD_SYS]
bs = []
for s in SEEDS_REPEAT:
    for code in BUD_SEED_SYS:
        st, jt = budget_inputs(code, "test", s)
        for cap in FPR_CAPS:
            pred = np.where(st >= BUD_TAU[(code, cap)], jt, BENIGN)
            bs.append({"seed": s, "system": code, "budget": cap, "tau": BUD_TAU[(code, cap)], **test_row(pred)})
BUDGET_SEEDS = pd.DataFrame(bs)
if len(BUDGET_SEEDS):
    BUDGET_SEEDS.to_csv(res("table_budget_seeds.csv"), index=False)
    cols = ["test_macro_f1", "test_minority_macro_f1", "test_infiltration_f1", "test_benign_fpr", "test_detection_rate"]
    BSUM = BUDGET_SEEDS.groupby(["system", "budget"])[cols].agg(["mean", "std"])
    BSUM.columns = [f"{a}_{b}" for a, b in BSUM.columns]
    pv = BUDGET_SEEDS.pivot_table(index=["budget", "seed"], columns="system", values="test_macro_f1")
    if PROPOSED in pv:
        for code in pv.columns:
            if code == PROPOSED:
                continue
            for cap in FPR_CAPS:
                d = (pv.loc[cap, PROPOSED] - pv.loc[cap, code]).dropna()
                BSUM.loc[(code, cap), "proposed_minus_this_mean"] = d.mean()
                BSUM.loc[(code, cap), "proposed_minus_this_std"] = d.std()
                BSUM.loc[(code, cap), "seeds_proposed_higher"] = int((d > 0).sum())
                BSUM.loc[(code, cap), "seeds"] = int(len(d))
    BSUM.to_csv(res("table_budget_seed_summary.csv"))
    print(BSUM.round(4).to_string())

# systems at FA_BUDGET in the results table
for c in BUDGET_ROWS:
    code = sys_code(c)
    if (code, FA_BUDGET) not in BUD_PRED:
        continue
    tau = BUD_TAU[(code, FA_BUDGET)]
    if code.startswith("S1-"):
        proba, label = FLAT[code[3:].lower()][1], FLAT_LABELS[code[3:].lower()]
    else:
        proba, label = composed_proba(p_test, COND_TEST[code]), code
        SYS_KEY[f"{code}-{FA_TAG}"] = (code, tau)
    add_result(f"{code}-{FA_TAG}", f"{label}, τ = {ftau(tau)} ({100 * FA_BUDGET:g} % false-alarm budget)",
               f"{100 * FA_BUDGET:g} % false-alarm budget", BUD_PRED[(code, FA_BUDGET)], proba, tau)
save_table()

def per_class_tables():
    global PC
    PC = pd.DataFrame({code: RESULTS[code]["per_class"].set_index("class")["f1"] for code in RESULTS})
    PC.insert(0, "test_flows", np.bincount(y_test, minlength=K)); PC.insert(0, "train_flows", train_counts)
    PC = PC.sort_values("train_flows", ascending=False); PC.to_csv(res("table_per_class_f1.csv"), index_label="class")
    pd.DataFrame({code: RESULTS[code]["per_class"].set_index("class")["precision"] for code in RESULTS}).to_csv(
        res("table_per_class_precision.csv"), index_label="class")
    pd.DataFrame({code: (1 - RESULTS[code]["per_class"].set_index("class")["f1"]) /
                        max((1 - RESULTS[code]["per_class"]["f1"]).sum(), 1e-12) for code in RESULTS}).to_csv(
        res("table_gap_share.csv"), index_label="class")

per_class_tables()
print(PC[["train_flows", "test_flows"] + [c for c in ["S1-STACK", f"S1-STACK-{FA_TAG}", "KARATAS-RF", "DSSTE-RF",
                                                     "T-SMOTE", "R-SMOTE", PROPOSED] if c in PC]].round(3).to_string())

## 15. Ablation chain and Stage-2 rejections
**Chain:** each row adds one change to the row above, from the v3 proposed system to the proposed system.
**Rejections:** at each benign-class system's τ (main rule and `FA_BUDGET`), how many gate false alarms Stage 2
returned to benign and how many gated attack flows it lost to benign, per class.

In [ ]:
CHAIN = [("T-CTGAN", "v3: published features, one CTGAN, global clipping, attack-only Stage 2, τ = 0.45"),
         ("T-CTGAN-PS", "+ Stage-2 features chosen on class-balanced attack flows"),
         ("T-CTGAN-PS-CC", "+ class-wise rounding and clipping of synthetic flows"),
         ("R-CTGAN-45", "+ benign class in Stage 2 (features re-selected with benign hard negatives)"),
         ("R-CTGAN", "+ τ chosen on validation")]
if PROPOSED == "R-CTGAN-PC":
    CHAIN.append(("R-CTGAN-PC", "one CTGAN per class instead of one conditional CTGAN"))
ch, prev = [], None
for step, (code, change) in enumerate(CHAIN, 1):
    if code not in TABLE.index:
        continue
    r = TABLE.loc[code]
    ch.append({"step": step, "code": code, "change": change, "tau": r["tau"], "macro_f1": r["macro_f1"],
               "minority_macro_f1": r["minority_macro_f1"], "infiltration_f1": r["infiltration_f1"],
               "macro_f1_excl_infiltration": r["macro_f1_excl_infiltration"], "benign_fpr": r["benign_fpr"],
               "attack_detection_rate": r["attack_detection_rate"],
               "delta_macro_f1": np.nan if prev is None else r["macro_f1"] - prev})
    prev = r["macro_f1"]
CHAIN_TAB = pd.DataFrame(ch); CHAIN_TAB.to_csv(res("table_ablation_chain.csv"), index=False)
print(CHAIN_TAB.drop(columns="change").round(4).to_string(index=False))

def rejection_row(code, key, tau):
    cond = COND_TEST[key]
    gated, j = p_test >= tau, codes(cond)[cond.argmax(1)]
    gb, ga = gated & tb, gated & tatt
    return {"code": code, "tau": tau, "benign_flagged_by_gate": int(gb.sum()),
            "returned_to_benign_by_stage2": int((j[gb] == BENIGN).sum()),
            "share_of_gate_false_alarms_removed": float((j[gb] == BENIGN).mean()) if gb.any() else np.nan,
            "attacks_passed_by_gate": int(ga.sum()), "attacks_sent_to_benign_by_stage2": int((j[ga] == BENIGN).sum()),
            "share_of_gated_attacks_lost": float((j[ga] == BENIGN).mean()) if ga.any() else np.nan}

rj, rjc = [], []
for code, (key, tau) in SYS_KEY.items():
    cond = COND_TEST[key]
    if cond.shape[1] != A + 1:
        continue
    rj.append(rejection_row(code, key, tau))
    if code == PROPOSED:
        gated, j = p_test >= tau, codes(cond)[cond.argmax(1)]
        gb, ga = gated & tb, gated & tatt
        for c in ATTACK_CODES:
            m = ga & (y_test == c)
            rjc.append({"class": CLASS_NAMES[c], "test_flows": int((y_test == c).sum()), "passed_by_gate": int(m.sum()),
                        "sent_to_benign_by_stage2": int((j[m] == BENIGN).sum()),
                        "benign_false_alarms_labelled_as_this_class": int((j[gb] == c).sum())})
REJ = pd.DataFrame(rj); REJ.to_csv(res("table_stage2_rejections.csv"), index=False)
REJC = pd.DataFrame(rjc); REJC.to_csv(res("table_stage2_rejections_by_class.csv"), index=False)
print(REJ.round(4).to_string(index=False)); print(REJC.to_string(index=False))
if "T-CTGAN" in SYS_KEY:     # where the v3 false alarms went, for comparison
    j3 = decide(p_test, COND_TEST["T-CTGAN"], TAU_MAIN)
    pd.Series(j3[tb & (j3 != BENIGN)]).map(dict(enumerate(CLASS_NAMES))).value_counts().rename(
        "v3_benign_false_alarms_by_label").to_csv(res("table_v3_false_alarm_labels.csv"))

## 16. Where the false alarms go (item 7)
For each pair in `FA_PAIRS` (attack-only Stage 2 against benign-class Stage 2), both at their `FA_BUDGET` operating
point from Section 14. Per attack class: the benign test flows labelled as that class, and the attack flows of that
class that end as benign, split into those the gate missed and those Stage 2 sent back to benign. The totals table
gives the gate false alarms and how many of them Stage 2 removed.

In [ ]:
def fa_destinations(code, tau):
    cond = COND_TEST[code]
    gated, j = p_test >= tau, codes(cond)[cond.argmax(1)]
    pred = np.where(gated, j, BENIGN)
    per = [{"class": CLASS_NAMES[c], "test_flows": int((y_test == c).sum()),
            "benign_labelled_as_class": int((tb & (pred == c)).sum()),
            "missed_by_gate": int(((y_test == c) & ~gated).sum()),
            "sent_to_benign_by_stage2": int(((y_test == c) & gated & (j == BENIGN)).sum())} for c in ATTACK_CODES]
    tot = {"benign_test_flows": int(tb.sum()), "benign_flagged_by_gate": int((tb & gated).sum()),
           "benign_returned_by_stage2": int((tb & gated & (j == BENIGN)).sum()),
           "final_benign_false_alarms": int((tb & (pred != BENIGN)).sum()),
           "attack_flows_missed_by_gate": int((tatt & ~gated).sum()),
           "attack_flows_sent_to_benign_by_stage2": int((tatt & gated & (j == BENIGN)).sum())}
    return per, tot

fa_rows, fa_tot = [], []
for a, b in FA_PAIRS:
    a, b = sys_code(a), sys_code(b)
    if (a, FA_BUDGET) not in BUD_TAU or (b, FA_BUDGET) not in BUD_TAU:
        continue
    for role, code in (("attack-only", a), ("benign-class", b)):
        tau = BUD_TAU[(code, FA_BUDGET)]
        per, tot = fa_destinations(code, tau)
        fa_rows += [{"pair": f"{a} vs {b}", "stage2": role, "system": code, "tau": tau, **r} for r in per]
        fa_tot.append({"pair": f"{a} vs {b}", "stage2": role, "system": code, "tau": tau, "budget": FA_BUDGET, **tot})
FADEST = pd.DataFrame(fa_rows); FADEST.to_csv(res("table_false_alarm_destinations.csv"), index=False)
FATOT = pd.DataFrame(fa_tot); FATOT.to_csv(res("table_false_alarm_totals.csv"), index=False)
if len(FADEST):
    print(FATOT.to_string(index=False))
    first = FADEST[FADEST.pair == FADEST.pair.iloc[0]]
    print(first.pivot(index="class", columns="system",
                      values=["benign_labelled_as_class", "missed_by_gate", "sent_to_benign_by_stage2"]).to_string())

## 17. Hard-negative sensitivity (item 3)
The proposed system is retrained with one hard-negative setting changed at a time: the score threshold
(`HN_SENS_TAUS`, cap fixed at `HN_CAP`) and the cap (`HN_SENS_CAPS`, threshold fixed at `HN_TAU`; `None` = every
eligible flow). Features, generator pool and gate stay fixed. Each setting has its own fit-part Stage 2, so τ is chosen
on validation (main rule and `FA_BUDGET`) and searched at or above the setting's own score threshold.

In [ ]:
def hn_code(hn):
    if hn == HN_MAIN:
        return PROPOSED
    return f"{PROPOSED}-HN{int(round(hn[0] * 100))}-" + ("ALL" if hn[1] is None else f"{hn[1] // 1000}K")

hs = []
if RUN_HN_SENS:
    HN_GRID = list(dict.fromkeys([(t, HN_CAP) for t in HN_SENS_TAUS] + [(HN_TAU, c) for c in HN_SENS_CAPS]))
    def n_eligible(t):
        return {part: int(np.sum((y_train[rows] == BENIGN) & (sc[rows] >= t)))
                for part, rows, sc in (("fit", fit_idx, P_OOF_FIT), ("full", all_idx, p_oof))}
    for hn in HN_GRID:
        eligible = n_eligible(hn[0])
        varied = "main setting" if hn == HN_MAIN else ("score threshold" if hn[1] == HN_CAP else "cap")
        # a cap that neither this setting nor the main one reaches gives the same training data as the main setting
        cap_inf = lambda c: np.inf if c is None else c
        same = hn != HN_MAIN and hn[0] == HN_TAU and all(e <= min(cap_inf(hn[1]), cap_inf(HN_CAP)) for e in eligible.values())
        k = PROPOSED if same else hn_code(hn)
        if same:
            varied += " (cap not reached: same model as the main setting)"
        if k not in VARIANTS:
            VARIANTS[k] = spec(PROPOSED_GEN, "per_stage_r", True, "v4", hn)
        if k not in COND_VAL:
            COND_VAL[k], _ = stage2(VARIANTS[k], "fit")
        if k not in COND_TEST:
            COND_TEST[k], S2_INFO[k] = stage2(VARIANTS[k], "full")
        for rule, cap in (("main rule", MAX_BENIGN_FPR), (f"{100 * FA_BUDGET:g} % budget", FA_BUDGET)):
            b = choose_tau(*two_inputs(k, "val"), cap, grid_for(k))
            st, jt = two_inputs(k, "test")
            pred = np.where(st >= b["tau"], jt, BENIGN)
            rr = rejection_row(k, k, b["tau"])
            hs.append({"setting": hn_code(hn), "varied": varied, "hn_tau": hn[0], "hn_cap": "all" if hn[1] is None else hn[1],
                       "eligible_fit": eligible["fit"], "eligible_full": eligible["full"],
                       "used_full": S2_INFO[k].get("hard_negative_rows"), "rule": rule, **b, **test_row(pred),
                       "share_of_gate_false_alarms_removed": rr["share_of_gate_false_alarms_removed"],
                       "share_of_gated_attacks_lost": rr["share_of_gated_attacks_lost"]})
HNS = pd.DataFrame(hs)
if len(HNS):
    HNS.to_csv(res("table_hn_sensitivity.csv"), index=False)
    print(HNS[["setting", "rule", "eligible_full", "used_full", "tau", "val_benign_fpr", "test_benign_fpr",
               "test_macro_f1", "test_infiltration_f1", "share_of_gate_false_alarms_removed",
               "share_of_gated_attacks_lost"]].round(4).to_string(index=False))

## 18. Statistics: McNemar, bootstrap, seed repeats
* McNemar compares per-flow correctness of the proposed system with every other system in the results table
  (Holm-corrected), and, per budget, the proposed system with every other system at the same budget.
* The paired bootstrap resamples test flows and recomputes macro-F1 for every system on the same resample.
* Seed repeats retrain the generator, SMOTE, the hard-negative subset and Stage 2 with each seed; gate and τ stay fixed.

In [ ]:
def mcnemar(ok_a, ok_b):
    b, c = int(np.sum(ok_a & ~ok_b)), int(np.sum(~ok_a & ok_b))
    if b + c == 0: return b, c, 1.0, "no discordant pairs"
    if b + c < 25: return b, c, binomtest(min(b, c), b + c, 0.5).pvalue, "exact binomial"
    return b, c, chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1), "chi-square, continuity corrected"

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i, k in enumerate(order):
        run = max(run, min(1.0, (len(p) - i) * p[k])); out[k] = run
    return out

st = []
for o in [c for c in RESULTS if c != PROPOSED]:
    b, c, p, test = mcnemar(RESULTS[PROPOSED]["pred"] == y_test, RESULTS[o]["pred"] == y_test)
    st.append({"proposed": PROPOSED, "compared_with": o, "proposed_right_other_wrong": b,
               "proposed_wrong_other_right": c, "p_value": p, "test": test})
STATS = pd.DataFrame(st); STATS["p_holm"] = holm(STATS.p_value); STATS.to_csv(res("table_mcnemar.csv"), index=False)

def bud_name(code, cap): return f"{code}@{100 * cap:g}%"
BOOT_PREDS = {c: RESULTS[c]["pred"] for c in RESULTS}
BOOT_PREDS.update({bud_name(code, cap): p for (code, cap), p in BUD_PRED.items()})

def paired_bootstrap(preds):
    """macro-F1 of every system on the same N_BOOT resamples of the test flows (resample = flow weights)."""
    h = hashlib.sha1()
    for c in sorted(preds):
        h.update(c.encode()); h.update(np.ascontiguousarray(preds[c]).tobytes())
    h.update(f"{N_BOOT}_{SEED}".encode())
    name = f"bootstrap_v5_{h.hexdigest()[:12]}.npz"
    if have(name):
        z = np.load(ckpt(name)); return {c: z[c] for c in preds}
    rng = np.random.default_rng(SEED); n = len(y_test)
    keys = {c: (y_test.astype(np.int64) * K + p).astype(np.int32) for c, p in preds.items()}
    out = {c: np.empty(N_BOOT) for c in preds}
    for i in range(N_BOOT):
        w = np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64)
        for c, kk in keys.items():
            cm = np.bincount(kk, weights=w, minlength=K * K).reshape(K, K)
            denom = cm.sum(0) + cm.sum(1)
            f = np.where(denom > 0, 2 * np.diag(cm) / np.maximum(denom, 1e-12), 0.0)
            out[c][i] = f[denom > 0].mean()
    np.savez(ckpt(name), **out)
    return out

BOOT = paired_bootstrap(BOOT_PREDS)
bt = []
for c in RESULTS:
    row = {"code": c, "macro_f1": TABLE.loc[c, "macro_f1"],
           "ci95_low": np.percentile(BOOT[c], 2.5), "ci95_high": np.percentile(BOOT[c], 97.5)}
    if c != PROPOSED:
        d = BOOT[PROPOSED] - BOOT[c]
        row.update({"proposed_minus_this_low": np.percentile(d, 2.5), "proposed_minus_this_high": np.percentile(d, 97.5),
                    "share_proposed_higher": float((d > 0).mean())})
    bt.append(row)
BOOTTAB = pd.DataFrame(bt); BOOTTAB.to_csv(res("table_bootstrap.csv"), index=False)
print(STATS.round(5).to_string(index=False)); print(BOOTTAB.round(4).to_string(index=False))

bst = []
for cap in FPR_CAPS:
    if (PROPOSED, cap) not in BUD_PRED:
        continue
    ok_p, part = BUD_PRED[(PROPOSED, cap)] == y_test, []
    for o in [c for c in BUD_SYS if c != PROPOSED]:
        b, c, p, test = mcnemar(ok_p, BUD_PRED[(o, cap)] == y_test)
        d = BOOT[bud_name(PROPOSED, cap)] - BOOT[bud_name(o, cap)]
        part.append({"budget": cap, "proposed": PROPOSED, "compared_with": o, "proposed_right_other_wrong": b,
                     "proposed_wrong_other_right": c, "p_value": p, "test": test,
                     "macro_f1_difference": float(np.mean(d)), "difference_ci95_low": np.percentile(d, 2.5),
                     "difference_ci95_high": np.percentile(d, 97.5), "share_proposed_higher": float((d > 0).mean())})
    for r, ph in zip(part, holm([r["p_value"] for r in part])):
        r["p_holm_within_budget"] = ph
    bst += part
BUDSTATS = pd.DataFrame(bst); BUDSTATS.to_csv(res("table_budget_statistics.csv"), index=False)
print(BUDSTATS.round(5).to_string(index=False))

seed_rows = []
for s in SEEDS_REPEAT:
    for code in seed_codes:
        key, tau = SYS_KEY[code]
        cond, _ = stage2(VARIANTS[key], "full", seed=s)
        pred = decide(p_test, cond, tau)
        f, _ = f1_vector(y_test, pred)
        seed_rows.append({"seed": s, "system": code, "tau": tau, "macro_f1": f.mean(),
                          "minority_macro_f1": f[MINORITY_FULL].mean(), "augmented_classes_f1": f[AUG_CODES].mean(),
                          "benign_fpr": float((pred[tb] != BENIGN).mean())})
if seed_rows:
    SEEDS = pd.DataFrame(seed_rows); SEEDS.to_csv(res("table_seed_repeats.csv"), index=False)
    summ = SEEDS.groupby("system")[["macro_f1", "minority_macro_f1", "augmented_classes_f1", "benign_fpr"]].agg(["mean", "std"])
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    pv = SEEDS.pivot(index="seed", columns="system", values="macro_f1")
    if PROPOSED in pv:
        for c in pv.columns:
            if c != PROPOSED:
                summ.loc[c, "proposed_minus_this_mean"] = (pv[PROPOSED] - pv[c]).mean()
                summ.loc[c, "proposed_minus_this_std"] = (pv[PROPOSED] - pv[c]).std()
                summ.loc[c, "seeds_proposed_higher"] = int((pv[PROPOSED] > pv[c]).sum())
    summ.to_csv(res("table_seed_summary.csv")); print(summ.round(4).to_string())

## 19. Threshold sensitivity (test split)
Describes the trade-off between benign false alarms and macro-F1; it is not used to choose anything. Two-stage
systems are shown for τ ≥ `HN_TAU`, the range the benign-class Stage 2 is trained for. For S1-STACK, τ is the
threshold on its attack score 1 − P(benign).

In [ ]:
sw = []
sweep_codes = [c for c in ["S1-STACK", "T-SMOTE", "T-CTGAN", "R-SMOTE", PROPOSED]
               if c in SYS_KEY or (c.startswith("S1-") and c[3:].lower() in FLAT)]
for code in sweep_codes:
    if code.startswith("S1-"):
        score, j = flat_inputs(FLAT[code[3:].lower()][1])
    else:
        score, j = two_inputs(SYS_KEY[code][0], "test")
    for t in np.round(np.arange(0.05, 0.951, 0.05), 2):
        if t < HN_TAU - 1e-9:
            continue
        pred = np.where(score >= t, j, BENIGN); f, _ = f1_vector(y_test, pred)
        sw.append({"system": code, "tau": t, "macro_f1": f.mean(), "minority_macro_f1": f[MINORITY_FULL].mean(),
                   "infiltration_f1": f[INFIL] if INFIL is not None else np.nan,
                   "detection_rate": float((pred[tatt] != BENIGN).mean()),
                   "benign_fpr": float((pred[tb] != BENIGN).mean())})
SWEEP = pd.DataFrame(sw); SWEEP.to_csv(res("table_threshold_sensitivity_test.csv"), index=False)
print(SWEEP[SWEEP.system == PROPOSED].round(4).to_string(index=False))

## 20. Training and inference time on this machine (item 5)
v3 and v4 times came from different runtimes. Here every system in `TIMING_SYSTEMS` (and every published method) is
timed on the current machine, recorded in `environment.json` → `machine`. A fit already made on this machine (in this
run or an earlier run on the same runtime type) is reused; anything else is refitted once for timing only, and the
results above do not change.
* **Single-stage:** fit on the training split; inference = `predict_proba` on every test flow.
* **Two-stage training:** gate + (benign-class systems) 3 out-of-fold gate fits for the hard negatives + generator
  (per-class CTGAN, if `TIME_GENERATOR`) + Stage-2 data preparation (SMOTE, pool, hard negatives) + Stage-2 fit.
* **Two-stage inference:** gate on every test flow + Stage 2 only on the flows the gate passes at the system's τ.
  Reported per test flow.

In [ ]:
TIMING_CACHE = {}

def timing_json(name, fn):
    """Load a same-machine timing record from the checkpoints, or run fn() and store its result."""
    path = ckpt(f"timing5_{name}.json")
    if name in TIMING_CACHE:
        return TIMING_CACHE[name]
    if os.path.exists(path):
        rec = load_json(path)
        if same_machine(rec):
            TIMING_CACHE[name] = rec
            return rec
    rec = {**fn(), **MACHINE}
    save_json(rec, path); TIMING_CACHE[name] = rec
    return rec

def time_flat(key):
    for s in [SEED] + [x for x in SEEDS_REPEAT if x != SEED]:
        base = FLAT_CKPT[key] + ("" if s == SEED else f"_s{s}")
        if os.path.exists(ckpt(base + ".json")):
            info = load_json(ckpt(base + ".json"))
            if same_machine(info):
                return {"train_seconds": info["train_seconds"], "predict_seconds": info["predict_seconds"],
                        "source": f"fit in a run on this machine (seed {s})"}
    def fn():
        model = FLAT_BUILDERS[key](SEED)
        with Timer() as t1:
            model.fit(X_train[F_PUB], y_train)
        with Timer() as t2:
            model.predict_proba(X_test[F_PUB])
        return {"train_seconds": t1.s, "predict_seconds": t2.s, "source": "refitted for timing"}
    return timing_json(f"flat_{key}", fn)

def time_gate():
    def fn():
        _, t, _ = gate_fit_score(all_idx, X_test)
        with Timer() as t_oof:
            gate_oof(all_idx)
        return {"train_seconds": t["train_seconds"], "predict_seconds": t["predict_seconds"],
                "oof_seconds": t_oof.s, "source": "refitted for timing"}
    return timing_json(f"gate_{GATE_TAG}", fn)

def time_generator(sp):
    if sp["method"] not in GAN_METHODS or not TIME_GENERATOR:
        return {"seconds": 0.0, "source": "none"}
    log = (GEN_LOG.get(sp["method"]) if sp["pool"] == "v4" else None) or {}
    if same_machine(log):
        return {"seconds": log["fit_seconds"] + log["sample_seconds"], "source": "fit in a run on this machine"}
    def fn():
        _, lg = (generate_pool if sp["pool"] == "v3" else generate_pool4)(sp["method"], SEED, timing_only=True)
        return {"seconds": lg["fit_seconds"] + lg["sample_seconds"], "fit_seconds": lg["fit_seconds"],
                "sample_seconds": lg["sample_seconds"], "source": "refitted for timing"}
    return timing_json(f"generator_{sp['pool']}_{sp['method']}", fn)

def time_two(code):
    key, tau = SYS_KEY[code]
    sp = VARIANTS[key]
    def fn():
        with Timer() as t_data:
            X, yc, _, _ = stage2_data(all_idx, sp, SEED, "full")
        stack = build_stack(int(yc.max()) + 1, SEED)
        with Timer() as t_fit:
            stack.fit(X, yc)
        gated = p_test >= tau
        with Timer() as t_pred:
            if gated.any():
                stack.predict_proba(X_test.loc[gated, FEATS[sp["feat"]]])
        return {"stage2_data_seconds": t_data.s, "stage2_train_seconds": t_fit.s, "stage2_predict_seconds": t_pred.s,
                "flows_to_stage2": int(gated.sum()), "tau": tau, "source": "refitted for timing"}
    return timing_json(f"stage2_{s2_name(sp, 'full', SEED)}_tau{ftau(tau)}", fn)

tm = []
if RUN_TIMING:
    n_test = len(y_test)
    for code in TIMING_SYSTEMS:
        code = sys_code(code)
        if code.startswith("S1-") and code[3:].lower() in FLAT:
            r = time_flat(code[3:].lower())
            tm.append({"code": code, "kind": "single-stage", "train_seconds": r["train_seconds"],
                       "inference_seconds": r["predict_seconds"],
                       "inference_us_per_flow": 1e6 * r["predict_seconds"] / n_test, "source": r["source"]})
        elif code in SYS_KEY:
            sp = VARIANTS[SYS_KEY[code][0]]
            g, gen, s2 = time_gate(), time_generator(sp), time_two(code)
            oof = g["oof_seconds"] if sp["reject"] else 0.0
            train = g["train_seconds"] + oof + gen["seconds"] + s2["stage2_data_seconds"] + s2["stage2_train_seconds"]
            infer = g["predict_seconds"] + s2["stage2_predict_seconds"]
            tm.append({"code": code, "kind": "two-stage", "tau": s2["tau"], "gate_train_seconds": g["train_seconds"],
                       "oof_gate_seconds": oof, "generator_seconds": gen["seconds"],
                       "stage2_data_seconds": s2["stage2_data_seconds"], "stage2_train_seconds": s2["stage2_train_seconds"],
                       "train_seconds": train, "gate_inference_seconds": g["predict_seconds"],
                       "stage2_inference_seconds": s2["stage2_predict_seconds"], "flows_to_stage2": s2["flows_to_stage2"],
                       "share_of_flows_to_stage2": s2["flows_to_stage2"] / n_test, "inference_seconds": infer,
                       "inference_us_per_flow": 1e6 * infer / n_test,
                       "source": f"gate: {g['source']}; generator: {gen['source']}; stage 2: {s2['source']}"})
    for code, (pred, proba, info) in PUB.items():
        if same_machine(info):
            tm.append({"code": code, "kind": "published method", "rebalancing_seconds": info["data_seconds"],
                       "train_seconds": info["data_seconds"] + info["train_seconds"],
                       "inference_seconds": info["predict_seconds"],
                       "inference_us_per_flow": 1e6 * info["predict_seconds"] / n_test,
                       "source": "fit in a run on this machine (rebalancing included)"})
TIMING = pd.DataFrame(tm)
if len(TIMING):
    for k_, v_ in MACHINE.items():
        TIMING[k_] = v_
    TIMING.to_csv(res("table_timing_same_machine.csv"), index=False)
    print(TIMING[["code", "train_seconds", "inference_us_per_flow", "source"]].round(2).to_string(index=False))
    for _, r in TIMING.iterrows():
        TABLE.loc[r["code"], "train_seconds_this_machine"] = r["train_seconds"]
        TABLE.loc[r["code"], "inference_us_per_flow_this_machine"] = r["inference_us_per_flow"]
    TABLE.to_csv(res("table_results.csv"))

## 21. Figures
Colour = augmentation (blue none, orange SMOTE, green CTGAN); dashed = attack-only Stage 2, solid = benign-class
Stage 2, dotted grey = single-stage.

In [ ]:
plt.rcParams.update({"font.size": 8.5, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e4e0", "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984",
                     "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "savefig.dpi": 300, "savefig.bbox": "tight"})
BLUE, ORANGE, GREEN, GRAY, LIGHT = "#2a78d6", "#eb6834", "#1baf7a", "#52514e", "#a9a8a3"
STYLE = {"S1-STACK": (GRAY, ":", "^", "Stacked ensemble, single-stage"),
         "S1-LGBM": (LIGHT, ":", "v", "LightGBM, single-stage"),
         "T-NONE": (BLUE, "--", "o", "None, attack-only S2"), "T-SMOTE": (ORANGE, "--", "s", "SMOTE, attack-only S2"),
         "T-CTGAN": (GREEN, "--", "D", "CTGAN, attack-only S2"), "R-NONE": (BLUE, "-", "o", "None, benign-class S2"),
         "R-SMOTE": (ORANGE, "-", "s", "SMOTE, benign-class S2"),
         PROPOSED: (GREEN, "-", "D", f"{NAMES[VARIANTS[PROPOSED]['method']]}, benign-class S2 (proposed)")}

# ablation chain
if len(CHAIN_TAB):
    f, ax = plt.subplots(figsize=(6.2, 0.42 * len(CHAIN_TAB) + 0.8))
    yy = np.arange(len(CHAIN_TAB))[::-1]
    ax.barh(yy, CHAIN_TAB.macro_f1, 0.6, color=[GREEN if c == PROPOSED else "#9fd9c3" for c in CHAIN_TAB.code])
    for y_, v in zip(yy, CHAIN_TAB.macro_f1):
        ax.text(v + 0.003, y_, f"{v:.3f}", va="center", fontsize=7.5)
    ax.set_yticks(yy, [f"{s}. {c}" for s, c in zip(CHAIN_TAB.step, CHAIN_TAB.code)])
    lo = max(0.0, CHAIN_TAB.macro_f1.min() - 0.08)
    ax.set_xlim(lo, min(1.0, CHAIN_TAB.macro_f1.max() + 0.04)); ax.set_xlabel("Macro-F1 (test)")
    ax.grid(axis="y", visible=False)
    f.tight_layout(); f.savefig(fig("fig_ablation_chain.png")); plt.close(f)

# false-alarm budgets (item 1): macro-F1 with seed spread, and test false alarms against the budget
show_b = [c for c in dict.fromkeys(sys_code(c) for c in BUDGET_ROWS) if c in BUD_SYS]
if show_b:
    f, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 3.0))
    x = 100 * np.array(FPR_CAPS)
    for code in show_b:
        col, ls, mk, lab = STYLE.get(code, (GRAY, "-", "o", code))
        b = BUDGET[BUDGET.system == code].sort_values("budget")
        if len(BUDGET_SEEDS) and code in set(BUDGET_SEEDS.system):
            g = BUDGET_SEEDS[BUDGET_SEEDS.system == code].groupby("budget").test_macro_f1
            a1.errorbar(x, g.mean().values, yerr=g.std().values, color=col, ls=ls, marker=mk, ms=4, lw=1.4,
                        capsize=2.5, label=lab + f" ({len(SEEDS_REPEAT)} seeds)")
        else:
            a1.plot(x, b.test_macro_f1, color=col, ls=ls, marker=mk, ms=4, lw=1.4, label=lab)
        a2.plot(x, 100 * b.test_benign_fpr, color=col, ls=ls, marker=mk, ms=4, lw=1.4)
    a2.plot(x, x, color="#0b0b0b", ls=(0, (1, 1.5)), lw=0.9, label="budget")
    for a in (a1, a2):
        a.set_xscale("log"); a.set_xticks(x, [f"{v:g}" for v in x]); a.minorticks_off()
        a.set_xlabel("False-alarm budget (%)")
    a1.set_ylabel("Macro-F1 (test)"); a2.set_ylabel("Benign flows flagged on test (%)")
    a2.legend(frameon=False, fontsize=7)
    h_, l_ = a1.get_legend_handles_labels()
    f.tight_layout(rect=(0, 0.2, 1, 1))
    f.legend(h_, l_, frameon=False, fontsize=7, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=2, handlelength=3.2)
    f.savefig(fig("fig_budget_comparison.png")); plt.close(f)

# per-class F1
bars = [c for c in [f"S1-STACK-{FA_TAG}", f"T-SMOTE-{FA_TAG}", f"{PROPOSED}-{FA_TAG}"] if c in PC.columns]
bar_style = {f"S1-STACK-{FA_TAG}": (LIGHT, STYLE["S1-STACK"][3]), f"T-SMOTE-{FA_TAG}": ("#f3b08f", STYLE["T-SMOTE"][3]),
             f"{PROPOSED}-{FA_TAG}": (GREEN, STYLE[PROPOSED][3])}
pcp = PC[PC.index != BENIGN_LABEL].iloc[::-1]
f, ax = plt.subplots(figsize=(6.2, 0.34 * len(pcp) + 0.9))
yy = np.arange(len(pcp)); h = 0.8 / max(len(bars), 1)
for i, c in enumerate(bars):
    ax.barh(yy + (len(bars) / 2 - i - 0.5) * h, pcp[c], h * 0.85, color=bar_style[c][0], label=bar_style[c][1])
ax.set_yticks(yy, [f"{n} ({t:,})" for n, t in zip(pcp.index, pcp.train_flows)])
ax.set_xlim(0, 1)
ax.set_xlabel(f"F1-score on the test split at the {100 * FA_BUDGET:g} % false-alarm budget; training flows in brackets")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.35, -0.11), ncol=1, fontsize=7.5)
f.savefig(fig("fig_per_class_f1.png")); plt.close(f)

# where the false alarms go (item 7), first pair
if len(FADEST):
    pair = FADEST.pair.iloc[0]; d = FADEST[FADEST.pair == pair]
    sys_a, sys_b = d[d.stage2 == "attack-only"].system.iloc[0], d[d.stage2 == "benign-class"].system.iloc[0]
    cls = [CLASS_NAMES[c] for c in ATTACK_CODES][::-1]
    da, db = d[d.system == sys_a].set_index("class").reindex(cls), d[d.system == sys_b].set_index("class").reindex(cls)
    f, (a1, a2) = plt.subplots(1, 2, figsize=(7.4, 0.3 * len(cls) + 1.3), sharey=True)
    yy = np.arange(len(cls)); hh = 0.38
    ca, cb = STYLE.get(sys_a, (ORANGE,))[0], STYLE.get(sys_b, (GREEN,))[0]
    a1.barh(yy + hh / 2, da.benign_labelled_as_class, hh, color=ca, label=f"{sys_a} (attack-only S2)")
    a1.barh(yy - hh / 2, db.benign_labelled_as_class, hh, color=cb, label=f"{sys_b} (benign-class S2)")
    a2.barh(yy + hh / 2, da.missed_by_gate, hh, color=ca)
    a2.barh(yy + hh / 2, da.sent_to_benign_by_stage2, hh, left=da.missed_by_gate, color=ca, alpha=0.45,
            hatch="////", edgecolor="white", lw=0)
    a2.barh(yy - hh / 2, db.missed_by_gate, hh, color=cb)
    a2.barh(yy - hh / 2, db.sent_to_benign_by_stage2, hh, left=db.missed_by_gate, color=cb, alpha=0.45,
            hatch="////", edgecolor="white", lw=0)
    for a in (a1, a2):
        a.set_xscale("symlog", linthresh=1); a.grid(axis="y", visible=False)
    a1.set_yticks(yy, cls)
    a1.set_xlabel("Benign test flows labelled as this class"); a2.set_xlabel("Attack flows of this class ending as benign")
    from matplotlib.patches import Patch
    a1.legend(frameon=False, fontsize=7, loc="lower right")
    a2.legend(handles=[Patch(facecolor=GRAY, label="missed by the gate"),
                       Patch(facecolor=GRAY, alpha=0.45, hatch="////", edgecolor="white", label="sent to benign by Stage 2")],
              frameon=False, fontsize=7, loc="lower right")
    f.suptitle(f"At the {100 * FA_BUDGET:g} % false-alarm budget", fontsize=8.5, color=GRAY)
    f.tight_layout(); f.savefig(fig("fig_false_alarm_destinations.png")); plt.close(f)

# hard-negative sensitivity (item 3)
if len(HNS):
    main = HNS[HNS.rule == "main rule"].copy()
    groups = [("score threshold", main[main.hn_cap.astype(str) == str(HN_CAP)].sort_values("hn_tau"),
               lambda r: f"{r.hn_tau:.2f}"),
              ("cap", main[np.isclose(main.hn_tau, HN_TAU)], lambda r: "all" if r.hn_cap == "all" else f"{int(r.hn_cap) // 1000}k")]
    f, axes = plt.subplots(1, 2, figsize=(6.6, 2.6), sharey=True)
    for a, (title, g, lab) in zip(axes, groups):
        if title == "cap":
            g = g.assign(_o=[1e12 if v == "all" else float(v) for v in g.hn_cap]).sort_values("_o")
        xs = np.arange(len(g))
        a.plot(xs, g.test_macro_f1, color=GREEN, marker="D", ms=4, lw=1.4, label="Macro-F1 (test)")
        a.plot(xs, g.test_minority_macro_f1, color=GREEN, marker="o", ms=3.5, lw=1.2, ls="--", label="Minority macro-F1")
        a.set_xticks(xs, [lab(r) for r in g.itertuples()]); a.set_xlabel(f"Hard-negative {title}")
        a2 = a.twinx(); a2.plot(xs, 100 * g.test_benign_fpr, color=GRAY, marker="s", ms=3.5, lw=1.0, ls=":",
                                label="Benign flows flagged (right axis)")
        a2.set_ylabel("Benign flows flagged (%)", color=GRAY); a2.grid(False); a2.spines["right"].set_visible(True)
    h1, l1 = axes[0].get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
    axes[0].set_ylabel("Macro-F1 (test)")
    f.tight_layout(rect=(0, 0.1, 1, 1))
    f.legend(h1 + h2, l1 + l2, frameon=False, fontsize=7, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0))
    f.savefig(fig("fig_hn_sensitivity.png")); plt.close(f)

# threshold sensitivity
f, (a1, a2) = plt.subplots(1, 2, figsize=(7.0, 2.8))
for code in SWEEP.system.unique():
    s = SWEEP[SWEEP.system == code]; col, ls, mk, lab = STYLE.get(code, (GRAY, "-", "o", code))
    a1.plot(s.tau, s.macro_f1, color=col, ls=ls, marker=mk, ms=3, lw=1.4, label=lab)
    a2.plot(s.tau, 100 * s.benign_fpr, color=col, ls=ls, marker=mk, ms=3, lw=1.4)
for a in (a1, a2):
    a.axvline(TAU_MAIN, color=GRAY, ls=":", lw=0.9); a.set_xlabel("Threshold τ")
a1.set_ylabel("Macro-F1 (test)"); a2.set_ylabel("Benign flows flagged (%)")
a1.legend(frameon=False, fontsize=7, handlelength=3.2)
f.tight_layout(); f.savefig(fig("fig_threshold_sensitivity.png")); plt.close(f)

# gate check
f, ax = plt.subplots(figsize=(4.4, 2.8))
for col_, c, ls, lab in (("fpr_validation_fit_gate", ORANGE, "-", "Validation, fit-part gate"),
                         ("fpr_validation_oof", BLUE, "--", "Validation, out-of-fold gate"),
                         ("fpr_test_full_gate", GRAY, "-", "Test, full gate")):
    ax.plot(GCHK.tau, 100 * GCHK[col_], color=c, ls=ls, marker="o", ms=3, lw=1.4, label=lab)
ax.set_xlabel("Gate threshold τ"); ax.set_ylabel("Benign flows flagged (%)"); ax.legend(frameon=False, fontsize=7, handlelength=3.2)
f.tight_layout(); f.savefig(fig("fig_gate_check.png")); plt.close(f)

# seed repeats (main rule)
if seed_rows:
    f, ax = plt.subplots(figsize=(4.6, 2.7))
    present = [c for c in seed_codes if c in SEEDS.system.unique()]
    for i, c in enumerate(present):
        v = SEEDS[SEEDS.system == c].macro_f1.values; col, _, mk, _ = STYLE.get(c, (GRAY, "-", "o", c))
        ax.scatter(np.full(len(v), i) + np.linspace(-0.08, 0.08, len(v)), v, color=col, marker=mk, s=22, zorder=3)
        ax.hlines(v.mean(), i - 0.22, i + 0.22, color="#0b0b0b", lw=1.2)
    ax.set_xticks(range(len(present)), present); ax.set_ylabel("Macro-F1 (test)"); ax.grid(axis="x", visible=False)
    f.tight_layout(); f.savefig(fig("fig_seed_repeats.png")); plt.close(f)

# synthetic-data quality (C2ST AUC; 0.5 = indistinguishable)
methods_q = list(QUALITY.method.unique())
f, axes = plt.subplots(1, len(methods_q), figsize=(1.9 * len(methods_q) + 1.6, 0.3 * len(AUG_CODES) + 1.0), sharey=True)
axes = np.atleast_1d(axes)
cls_order = [CLASS_NAMES[c] for c in AUG_CODES]
for a, m in zip(axes, methods_q):
    q = QUALITY[QUALITY.method == m].set_index("class").reindex(cls_order)
    a.scatter(q.c2st_auc, np.arange(len(q)), color=GREEN if m == VARIANTS[PROPOSED]["method"] else GRAY, s=20, zorder=3)
    a.axvline(0.5, color="#8a8984", ls=":", lw=0.9); a.set_xlim(0.45, 1.02); a.set_title(NAMES.get(m, m), fontsize=8.5)
    a.set_xlabel("C2ST AUC"); a.grid(axis="y", visible=False)
axes[0].set_yticks(np.arange(len(cls_order)), cls_order); axes[0].invert_yaxis()
f.tight_layout(); f.savefig(fig("fig_synthetic_quality_c2st.png")); plt.close(f)
print("figures saved to", FIG_DIR)

## 22. Export

In [ ]:
with pd.ExcelWriter(res(f"tables_v5_{DATASET}.xlsx")) as xw:
    used = set()
    for name in ["table_results", "table_budget_comparison", "table_budget_seeds", "table_budget_seed_summary",
                 "table_budget_statistics", "table_published_settings", "table_false_alarm_destinations",
                 "table_false_alarm_totals", "table_hn_sensitivity", "table_timing_same_machine", "table_ablation_chain",
                 "table_stage2_rejections", "table_stage2_rejections_by_class", "table_per_class_f1",
                 "table_per_class_precision", "table_gap_share", "table_thresholds", "table_gate", "table_gate_check",
                 "table_stage2_true_attacks", "table_synthetic_quality", "table_synthetic_quality_summary",
                 "table_generator_timing", "table_mcnemar", "table_bootstrap", "table_seed_repeats",
                 "table_seed_summary", "table_threshold_sensitivity_test", "table_v3_false_alarm_labels",
                 "table_class_distribution", "table_feature_selection", "table_feature_selection_stage2_benign"]:
        if os.path.exists(res(name + ".csv")):
            sheet = name.replace("table_", "")[:31]
            while sheet in used:
                sheet = sheet[:29] + f"_{len(used) % 10}"
            used.add(sheet)
            pd.read_csv(res(name + ".csv")).to_excel(xw, sheet_name=sheet, index=False)

def records(df): return df.to_dict(orient="records") if isinstance(df, pd.DataFrame) and len(df) else []
save_json({"dataset": DATASET, "proposed": PROPOSED, "fpr_margin": FPR_MARGIN, "tau_main": TAU_MAIN,
           "tau_published_rule": TAU_PUB, "tau_validation": TAU_VAL, "gate_check": GATE_CHECK,
           "hard_negatives": HN_COUNTS, "stage2_features": FEATS["per_stage_r"],
           "augmented_classes": [CLASS_NAMES[c] for c in AUG_CODES], "machine": MACHINE, "prep": PREP,
           "results": {c: RESULTS[c]["metrics"] for c in RESULTS}, "budget": records(BUDGET),
           "hn_sensitivity": records(HNS), "timing": records(TIMING), "false_alarm_totals": records(FATOT)},
          res("RESULTS_SUMMARY.json"))
print("Results archive:", shutil.make_archive(os.path.join(OUT_DIR, f"results_v5_{DATASET}"), "zip", RES_DIR))